# Tutorial 9.2 · Inverse Design of a Compact $90^\circ$ Waveguide Bend (`ceviche_waveguide_bend`)

**Chapter 9 — Inverse Design**

Conventional dielectric waveguide bends require a radius of curvature of several
wavelengths to avoid radiative bending loss. Using topology optimization driven by
Wakis's exact discrete adjoint (`wakis.inverse`) and the `invrs-wakis` bridge, a
$90^\circ$ silicon waveguide bend can be compressed into a $1.6\,\mu\text{m} \times 1.6\,\mu\text{m}$
design region ($160\times 160$ pixels at $10\text{ nm}$ resolution inside a
$3.12\,\mu\text{m} \times 3.12\,\mu\text{m}$ simulation domain) while maintaining
$>96.1\%$ transmission ($-0.172\text{ to }-0.154\text{ dB}$) and $<-43.4\text{ dB}$
reflection across all 6 O-band wavelengths ($\lambda \in \{1265, 1270, 1275, 1285, 1290, 1295\}\text{ nm}$)
and strictly satisfying the $8\text{ px}$ ($80\text{ nm}$) minimum solid width and void spacing target.

### Learning goals

1. **The `invrs-gym` `ceviche_waveguide_bend` specification**: understand the
   $312\times 312$-pixel ($10\text{ nm}$ grid) geometry, 6-wavelength S-parameter objective,
   target transmission ($[-0.5, 0.0]\text{ dB}$) and reflection ($\le -20\text{ dB}$) windows,
   and the minimum solid width / void spacing constraint ($8 / 8\text{ px} = 80\text{ nm}$).
2. **Two-stage `FeasibleDensity` + Straight-Through Estimator (STE) optimizer stack**:
   - **Stage 1**: continuous density L-BFGS-B across $\beta = 4 \to 8 \to 16 \to 32$
     (30 + 30 + 25 + 20 = 105 steps) with a $R = 8\text{ px}$ conic filter and
     Wang–Lazarov–Sigmund $\tanh$ projection.
   - **Stage 2**: `imageruler` morphological opening/closing brush repair enforcing
     $8 / 8\text{ px}$ minimum solid width and void spacing by construction, coupled
     with 40 steps of Adam ($\text{lr} = 0.02$) Straight-Through Estimator (STE)
     refinement at $\beta = 64$ and `CandidatePool` selection (`ste_beta64` step 34).
3. **Wakis time-domain re-simulation & `ceviche` FDFD cross-validation**: rebuild
   the `ceviche_waveguide_bend` simulation directly from the stored Round-2 artifacts
   (`data/round2_bend/fields_compact.npz` and `design.npy`) using Wakis's
   `PolarizedModeSource`, `PolarizedModeMonitor`, and `match_fdfd=True` temporal
   dispersion pre-warping, and verify that Wakis reproduces the stored 67-wavelength
   dense spectrum (`dense_S`) and aligns with `ceviche` FDFD to
   $\max|\Delta|S|^2| = 3.77\times 10^{-4}$ on the 10 nm grid ($1.03\times 10^{-4}$ on the 40 nm grid).
4. **Like-for-like leaderboard & multi-seed comparison**: compare our Round-2
   official `eval_metric = +0.6427` at `8 / 8 px` (and second seed `+0.5793` at
   `8 / 8 px`, run-to-run spread $\Delta = 0.0634$, Claim C55) against our Round-1
   pre-STE runs (`+0.3150` and `-0.0355` when evaluated at 10 nm) and the `invrs-gym`
   leaderboard solution (`-1.2504` at `9 / 7 px`, which violates both the $8\text{ px}$
   void spacing constraint and the $-20\text{ dB}$ reflection ceiling).
5. **Pedagogical toy problem (separate from the challenge)**: run a compact,
   clearly labelled `"toy problem (not the challenge)"` optimization in Wakis so
   learners can execute the two-stage adjoint + morphological STE loop in seconds on CPU.

### Simulation setup

The setup figures below display the $3.12\,\mu\text{m}\times 3.12\,\mu\text{m}$ ($312\times 312$ cells at $\Delta x = 10\text{ nm}$, surrounded by 20-cell / $200\text{ nm}$ CPML on $\pm x, \pm y$) `ceviche_waveguide_bend` domain, showing the Port 1 $\text{TM}_0$ `PolarizedModeSource` and $S_{11}$ `PolarizedModeMonitor`, the orthogonal Port 2 $S_{21}$ objective `PolarizedModeMonitor`, and the hatched $1.60\,\mu\text{m}\times 1.60\,\mu\text{m}$ ($160\times 160\text{ px}$) design region $\Omega_d$ for both the initial greyscale density (`_initial`) and the Round-2 STE binary design (`_final` / base).

![Initial setup geometry](figures/00a_setup_geometry_initial.png)
![Initial mesh and excitation](figures/00b_setup_mesh_and_excitation_initial.png)
![Final setup geometry](figures/00a_setup_geometry_final.png)
![Final mesh and excitation](figures/00b_setup_mesh_and_excitation_final.png)
![Setup geometry](figures/00a_setup_geometry.png)
![Setup mesh and excitation](figures/00b_setup_mesh_and_excitation.png)

| Category | Component / Face | Specification |
| :--- | :--- | :--- |
| Domain | Size & bounds | Lx × Ly × Lz = 3.12 µm × 3.12 µm × 10 nm (z ∈ [-5 nm, 5 nm]) |
| Domain | Grid cells | Nx × Ny × Nz = 312 × 312 × 1 (97344 cells) |
| Discretization | Δx & resolution | Δ_min = 10 nm, 36.1 cells/λ (in n_max = 3.50), 8.0 cells across feature (80 nm) |
| Boundary | -x face | CPML (20 cells, 200 nm) |
| Boundary | +x face | CPML (20 cells, 200 nm) |
| Boundary | -y face | CPML (20 cells, 200 nm) |
| Boundary | +y face | CPML (20 cells, 200 nm) |
| Boundary | -z face | Periodic |
| Boundary | +z face | Periodic |
| Material | Background | SiO₂ cladding (ε = 2.25, n = 1.50) (isotropic) |
| Material | Region 1: Input Si waveguide (ε = 12.25, n = 3.50, w = 400 nm) | ε_r = 12.25, μ_r = 1.00, model = Isotropic dielectric, x ∈ [5 nm, 920 nm] (d = 915 nm) |
| Material | Region 2: Output Si waveguide (ε = 12.25, n = 3.50, w = 400 nm) | ε_r = 12.25, μ_r = 1.00, model = Isotropic dielectric, x ∈ [1.52 µm, 1.92 µm] (d = 400 nm) |
| Material | Region 3: Design region Ω_d (160×160 px, 1.60 µm × 1.60 µm, ε ∈ {2.25, 12.25}) | ε_r = 12.25, μ_r = 1.00, model = Isotropic dielectric, x ∈ [920 nm, 2.52 µm] (d = 1.60 µm) |
| Source | 1 · PolarizedModeSource | pos: x = 0.25 µm, y ∈ [1.00, 2.44] µm, pol: TM₀ (E_z), pulse: f₀ = 234.4 THz, Δf = 28.13 THz |
| Monitor | 2 · PolarizedModeMonitor | Port 1 PolarizedModeMonitor (S₁₁ reflection, TM₀ E_z), pos: x = 0.30 µm, y ∈ [1.00, 2.44] µm, freqs: 6 target channels λ ∈ {1265..1295} nm (67 dense λ ∈ [1245, 1315] nm) |
| Monitor | 3 · PolarizedModeMonitor | Port 2 Objective PolarizedModeMonitor (S₂₁ transmission, TM₀ E_z), pos: x ∈ [1.00, 2.44] µm, y = 0.30 µm, freqs: 6 target channels λ ∈ {1265..1295} nm (67 dense λ ∈ [1245, 1315] nm) |
| Variation | Initial design (_initial) | Stage-1 iter 0 initial greyscale density (mean ρ ≈ 0.49, β = 4) in 160 × 160 design box Ω_d (1.60 µm × 1.60 µm) |
| Variation | Final Round-2 design (_final) | Stage-2 STE binary design (ste_beta64:34, 8/8 px = 80 nm, official eval_metric = +0.6427) from data/round2_bend/design.npy |
| Variation | Quick re-simulation (task_1) | 2× coarsened 20 nm grid (156 × 156 × 1, 18.1 cells/λ_min) vs stored 10 nm reference |

---

## Physics, Evaluation Metric & Optimizer Stack

### 1. Multi-wavelength S-parameter objective and `invrs-gym` `eval_metric`

Adjoint optimization of compact integrated-photonic waveguide routing components
dates back to Lalau-Keraly, Bhargava, Miller & Yablonovitch (*Opt. Express* **21**,
21693, 2013; [Yale Miller Group](https://millergroup.yale.edu)), who showed that
shape and level-set sensitivities of electromagnetic waveguide devices can be
evaluated with a single forward and adjoint solve per iteration. To compare inverse-design
algorithms on a uniform footing, Chen et al. (*J. Opt. Soc. Am. B* **41**, A161, 2024)
introduced the community `photonics-opt-testbed` suite—pairing standardized test
problems with independent cross-solver evaluation and `imageruler` minimum-length-scale
verification—which `invrs-gym` (Schubert et al., arXiv:`2410.24132`; Hughes et al.,
*ACS Photonics* 2019) packages together with the `ceviche` integrated-photonics challenges.

In the `ceviche_waveguide_bend` challenge, a silicon strip waveguide of width
$w_{\text{wg}} = 400\text{ nm}$ ($40\text{ px}$ at $\Delta l = 10\text{ nm}$, slab
permittivity $\varepsilon_{\text{slab}} = 12.25 = 3.5^2$ in $\text{SiO}_2$ cladding
$\varepsilon_{\text{clad}} = 2.25 = 1.5^2$) enters from Port 1 on the left
($x = 0.245\,\mu\text{m}, y = 1.72\,\mu\text{m}$, port window $1.44\,\mu\text{m}$)
and exits orthogonally downward through Port 2 at the bottom
($x = 1.72\,\mu\text{m}, y = 0.245\,\mu\text{m}$). The two waveguides meet inside
a square design region $\Omega_d = [0.92, 2.52]\,\mu\text{m} \times [0.92, 2.52]\,\mu\text{m}$
($160\times 160$ pixels), surrounded by a $200\text{ nm}$ ($20\text{ px}$) absorptive layer.

Exciting the fundamental TM ($E_z$) guided mode at Port 1 with complex incident
amplitude $a_1(\lambda_k)$ produces outgoing mode amplitudes $b_1(\lambda_k)$
(back-reflection at Port 1) and $b_2(\lambda_k)$ (transmission into Port 2) across
the $N_\lambda = 6$ target wavelengths
$\lambda_k \in \{1265, 1270, 1275, 1285, 1290, 1295\}\text{ nm}$:

$$
S_{11}(\lambda_k) = \frac{b_1(\lambda_k)}{a_1(\lambda_k)}, \qquad
S_{21}(\lambda_k) = \frac{b_2(\lambda_k)}{a_1(\lambda_k)}.
$$

To eliminate leapfrog temporal dispersion when comparing Wakis time-domain FIT
with `ceviche` frequency-domain FDFD on the identical Yee grid ($r = 1$),
`match_fdfd=True` samples the running DFT monitors at the pre-warped frequency

$$
f_{\text{sim}}(\lambda_k) = \frac{1}{\pi \Delta t}\arcsin\!\bigl(\pi f_{\text{target}}(\lambda_k)\Delta t\bigr),
\qquad \text{so that}\quad \omega_t(f_{\text{sim}}) = \frac{2}{\Delta t}\sin(\pi f_{\text{sim}}\Delta t) = 2\pi f_{\text{target}}.
$$

The `invrs-gym` evaluator defines linear power windows for transmission
$[T_{\text{lo}}, T_{\text{hi}}] = [10^{-0.5/10}, 1.0] \approx [0.89125, 1.0]$
($[-0.5, 0.0]\text{ dB}$) and reflection
$[R_{\text{lo}}, R_{\text{hi}}] = [0.0, 10^{-20/10}] = [0.0, 0.01]$ ($\le -20\text{ dB}$).
The official scalar `eval_metric` is the worst-case normalized window margin
across all 6 wavelengths and both ports:

$$
\text{eval\_metric} = \min_{k \in \{1,\dots,6\}} \min\!\left(
\frac{|S_{21}(\lambda_k)|^2 - T_{\text{lo}}}{T_{\text{hi}} - T_{\text{lo}}},\;
\frac{R_{\text{hi}} - |S_{11}(\lambda_k)|^2}{R_{\text{hi}} - R_{\text{lo}}}
\right).
$$

Thus $\text{eval\_metric} > 0$ if and only if every wavelength channel satisfies
both $|S_{21}|^2 > -0.5\text{ dB}$ and $|S_{11}|^2 < -20\text{ dB}$.

### 2. `FeasibleDensity` parameterization & Straight-Through Estimator (STE)

Standard conic filtering + $\tanh$ projection (Wang, Lazarov & Sigmund, 2011) at
finite $\beta$ leaves narrow solid bridges or void crevices when thresholded at
$\eta = 0.5$. In Round 2 (`invrs_wakis.lengthscale.FeasibleDensity`), the latent
density is symmetrized across the $45^\circ$ bend diagonal, convolved with a conic
hat kernel of radius $R = 8\text{ px}$, and projected via $\tanh(\beta(\tilde{\rho}-\eta))$.
Minimum solid width $d_{\text{solid}} = 8\text{ px}$ and void spacing
$d_{\text{void}} = 8\text{ px}$ ($80\text{ nm}$) are **guaranteed** by morphological
`repair`: alternating binary opening $\mathcal{O}_{k_w}$ and closing $\mathcal{C}_{k_s}$
with `imageruler` circular structuring elements of diameter $8\text{ px}$ (plus
iterative dilation of any remaining flagged interior violation pixels) while
re-imposing diagonal symmetry and fixed border pixels until `imageruler` reports
$\ge 8 / 8\text{ px}$.

In Stage 2 (`ste_beta64`, 40 Adam steps at $\beta = 64, \text{lr} = 0.02$), the
optimizer evaluates the repaired binary design in the forward pass and pulls the
exact Wakis adjoint sensitivity back through the smooth $\beta = 64$ projection:

$$
\bar{\boldsymbol{\rho}}_{\text{STE}} = \bar{\boldsymbol{\rho}}_{\beta=64}
+ \text{stop\_gradient}\!\bigl(\text{repair}(\bar{\boldsymbol{\rho}}_{\beta=64}) - \bar{\boldsymbol{\rho}}_{\beta=64}\bigr).
$$

### 3. Summary of Verified `ceviche_waveguide_bend` Benchmark Results

All numbers below come directly from `data/round2_bend/summary.json`,
`data/leaderboard_rescore.json`, and `data/round1/*.json`:

| Entry | Source File | Official `eval_metric` | Wakis $r=1$ `eval_metric` | Min Solid / Void (`px` at 10 nm) | In-Band $|S_{21}|^2$ [dB] | Max $|S_{11}|^2$ [dB] |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: |
| **Wakis Round-2 (Best, Seed 1 rerun)** | `round2_bend/summary.json` | **$+0.6427$** | $+0.6413$ (modal) / $+0.6412$ (cev) | **$8 / 8\text{ px}$** (Feasible) | $-0.172\text{ to }-0.154$ | $-43.44$ |
| **Wakis Round-2 (Seed 1 initial)** | `leaderboard_rescore.json` | **$+0.5793$** | — | **$8 / 8\text{ px}$** (Feasible) | $-0.203\text{ to }-0.188$ | $-45.10$ |
| **Wakis Round-1 (Seed 1, 40 nm opt)** | `round1/...seed1__summary.json` | $+0.3150$ (10 nm) / $+0.5208$ (40 nm) | $+0.5198$ (40 nm) | $2 / 1\text{ px}$ at 40 nm (Violates) | $-0.336\text{ to }-0.247$ (10 nm) | $-24.12$ (10 nm) |
| **Wakis Round-1 (Seed 0, 40 nm opt)** | `round1/...seed0__summary.json` | $-0.0355$ (10 nm) / $+0.3801$ (40 nm) | $+0.3814$ (40 nm) | $3 / 4\text{ px}$ at 40 nm (Pre-STE) | $-0.519\text{ to }-0.260$ (10 nm) | $-25.84$ (10 nm) |
| **`invrs-gym` Leaderboard Best** | `leaderboard_rescore.json` | **$-1.2504$** | — | $9 / 7\text{ px}$ (Violates void $7<8$) | $-0.500\text{ to }-0.389$ | $-16.48$ (Violates $>-20$) |

In [ ]:
from __future__ import annotations

import argparse
import json
from pathlib import Path
import time

import matplotlib
import numpy as np
from scipy.constants import c as C0
from scipy.constants import epsilon_0 as EPS_0
from scipy.ndimage import binary_closing, binary_erosion, binary_opening
from scipy.signal import convolve2d

import wakis
from wakis.inverse import AdjointSimulation, MaterialGrid
from wakis.mode_solver import ModeMonitor, ModeSource, solve_modes
from wakis.sources import CurrentSource, GaussianSource

TUTORIAL_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
COMMITTED_DATA_DIR = TUTORIAL_DIR / "data"
ROUND2_DIR = COMMITTED_DATA_DIR / "round2_bend"
ROUND1_DIR = COMMITTED_DATA_DIR / "round1"
LEADERBOARD_PATH = COMMITTED_DATA_DIR / "leaderboard_rescore.json"
DATA_DIR = COMMITTED_DATA_DIR
FIG_DIR = TUTORIAL_DIR / "figures"
STUDY_DIR = TUTORIAL_DIR.parents[2] / "studies" / "adjoint-memory"

# Official invrs-gym ceviche_waveguide_bend target wavelengths [nm] and window bounds
CHALLENGE_WAVELENGTHS_NM = np.array([1265.0, 1270.0, 1275.0, 1285.0, 1290.0, 1295.0], dtype=np.float64)
T_LOWER_BOUND = float(np.float32(10.0 ** (-0.5 / 10.0)))  # -0.5 dB -> 0.8912509
T_UPPER_BOUND = 1.0  # 0.0 dB
R_LOWER_BOUND = 0.0
R_UPPER_BOUND = float(np.float32(10.0 ** (-20.0 / 10.0)))  # -20.0 dB -> 0.01


def get_task_table(quick: bool = False) -> list[dict]:
    """Return the task table for Tutorial 9.2 with verified resolution metrics."""
    # Challenge grid: lambda_min = 1265 nm, n_slab = 3.5 -> lambda_slab = 361.43 nm.
    # At 10 nm (stride=1): cpw = 36.14, feature = 80 nm = 8.0 cells.
    # At 20 nm (stride=2): cpw = 18.07, feature = 80 nm = 4.0 cells.
    cpw_10nm = 1265.0 / (3.5 * 10.0)
    stride_resim = 2 if quick else 1
    cpw_resim = 1265.0 / (3.5 * (10.0 * stride_resim))
    cells_feat_resim = 8.0 / float(stride_resim)
    cpw_toy = 28.0 / float(np.sqrt(6.0))
    return [
        {
            "index": 0,
            "name": "round2_trace_and_length_scale",
            "description": (
                "Load the real Round-2 optimization run from data/round2_bend/ (design.npy, "
                "log.jsonl, snapshots_thinned.npz, summary.json) and verify its 8/8 px (80 nm) "
                "imageruler-style minimum solid width and void spacing."
            ),
            "cpw": cpw_10nm,
            "cells_across_feature": 8.0,
            "target_px": 8,
            "demo_n_design": 32 if quick else 40,
            "demo_target_px": 4,
        },
        {
            "index": 1,
            "name": "wakis_resimulation_of_round2_design",
            "description": (
                "Rebuild the ceviche_waveguide_bend geometry from data/round2_bend/fields_compact.npz "
                "and re-simulate our stored Round-2 design (data/round2_bend/design.npy) with Wakis "
                "across all 67 dense wavelengths (1245..1315 nm) and 6 target channels."
            ),
            "cpw": cpw_resim,
            "cells_across_feature": cells_feat_resim,
            "stride": stride_resim,
            "fwidth": 0.12,
            "run_time": 1.8e-13,
        },
        {
            "index": 2,
            "name": "leaderboard_and_round1_comparison",
            "description": (
                "Compile and verify official gym eval_metric scores, S-parameter spectra, and "
                "length scales from data/round2_bend/summary.json (+0.6427 at 8/8 px), "
                "data/leaderboard_rescore.json (+0.5793 at 8/8 px vs leaderboard -1.2504 at 9/7 px), "
                "and data/round1/*.json."
            ),
            "cpw": cpw_10nm,
            "cells_across_feature": 8.0,
        },
        {
            "index": 3,
            "name": "toy_bend_teaching_optimization",
            "description": (
                "Pedagogical toy problem (not the challenge): run a small live 2D waveguide bend "
                "two-stage density (beta-continuation) + morphological brush STE optimization in "
                "Wakis and save to data/toy_bend_demo.npz."
            ),
            "cpw": cpw_toy,
            "cells_across_feature": 6.0,
            "n_grid": 40 if quick else 44,
            "n_design": 16 if quick else 18,
            "Nt": 360 if quick else 420,
            "n_iters_stage1": 8 if quick else 12,
            "n_iters_stage2": 4 if quick else 6,
            "brush_px": 3,
        },
        {
            "index": 4,
            "name": "optional_invrs_gym_rescore",
            "description": (
                "Optional official invrs-gym / imageruler evaluation of data/round2_bend/design.npy "
                "when invrs_gym is installed in the Python environment."
            ),
            "cpw": cpw_10nm,
            "cells_across_feature": 8.0,
        },
    ]


def circular_brush(diameter_px: int) -> np.ndarray:
    """Create an imageruler-compatible circular structuring element of diameter ``diameter_px`` [px]."""
    d = max(1, int(diameter_px))
    c = 0.5 * (d - 1)
    ax = np.arange(d, dtype=np.float64) - c
    xx, yy = np.meshgrid(ax, ax, indexing="ij")
    return (np.hypot(xx, yy) <= 0.5 * d + 1e-9).astype(bool)


def morphological_repair(binary_arr: np.ndarray, diameter_px: int) -> np.ndarray:
    """Apply multi-scale closing(opening(B)) with circular brushes up to ``diameter_px`` [px]."""
    b2 = np.asarray(binary_arr[:, :, 0] if binary_arr.ndim == 3 else binary_arr, dtype=bool)
    pad = max(2, int(diameter_px) + 2)
    b_pad = np.pad(b2, pad, mode="edge")
    for m in range(2, int(diameter_px) + 1):
        brush = circular_brush(m)
        b_pad = binary_closing(binary_opening(b_pad, structure=brush), structure=brush)
    for m in range(int(diameter_px), 1, -1):
        brush = circular_brush(m)
        b_pad = binary_closing(binary_opening(b_pad, structure=brush), structure=brush)
    out = b_pad[pad:-pad, pad:-pad].astype(np.float64)
    return out[:, :, None] if binary_arr.ndim == 3 else out


def count_interior_violations(
    binary_arr: np.ndarray,
    diameter_px: int,
     surrounding_window: np.ndarray | None = None,
    border: int = 0,
) -> tuple[int, int]:
    """Count interior solid and void morphological violation pixels at brush diameter ``diameter_px``.

    Matches the ``imageruler`` convention in which 1-pixel staircase corners on curved
    interfaces (outside the 4-connected eroded interior) are excluded from thin-feature
    violations.
    """
    b2 = np.asarray(binary_arr[:, :, 0] if binary_arr.ndim == 3 else binary_arr) >= 0.5
    d = max(1, int(diameter_px))
    if d <= 1:
        return 0, 0
    if surrounding_window is not None and border > 0:
        win = np.asarray(surrounding_window) >= 0.5
        sl = slice(border, -border)
    else:
        pad = d + 2
        win = np.pad(b2, pad, mode="edge")
        sl = slice(pad, -pad)

    cross = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=bool)
    inner_solid = binary_erosion(win, structure=cross)[sl, sl]
    inner_void = binary_erosion(~win, structure=cross)[sl, sl]
    brush = circular_brush(d)
    opened = binary_opening(win, structure=brush)[sl, sl]
    closed = binary_closing(win, structure=brush)[sl, sl]
    v_solid = int(np.sum((opened != b2) & inner_solid))
    v_void = int(np.sum((closed != b2) & inner_void))
    return v_solid, v_void


def measure_length_scales_px(
    binary_arr: np.ndarray,
    max_px: int = 12,
    surrounding_window: np.ndarray | None = None,
    border: int = 0,
) -> tuple[int, int]:
    """Measure minimum solid width and minimum void spacing [px] via circular brush opening/closing."""
    b2 = np.asarray(binary_arr[:, :, 0] if binary_arr.ndim == 3 else binary_arr) >= 0.5
    if np.all(b2) or not np.any(b2):
        return max_px, max_px

    min_solid = 1
    for d in range(2, max_px + 1):
        v_s, _ = count_interior_violations(b2, d, surrounding_window=surrounding_window, border=border)
        if v_s == 0:
            min_solid = d
        else:
            break

    min_void = 1
    for d in range(2, max_px + 1):
        _, v_v = count_interior_violations(b2, d, surrounding_window=surrounding_window, border=border)
        if v_v == 0:
            min_void = d
        else:
            break

    return int(min_solid), int(min_void)


def compute_bend_eval_metric(
    s21_sq: np.ndarray,
    s11_sq: np.ndarray,
    t_lower_bound: float = T_LOWER_BOUND,
    t_upper_bound: float = T_UPPER_BOUND,
    r_lower_bound: float = R_LOWER_BOUND,
    r_upper_bound: float = R_UPPER_BOUND,
) -> float:
    """Compute the exact ``invrs-gym`` window evaluation metric from ``|S21|^2`` and ``|S11|^2``.

    Parameters
    ----------
    s21_sq : ndarray
        Power transmission ``|S21(lambda_k)|^2`` across the target wavelengths [dimensionless].
    s11_sq : ndarray
        Power reflection ``|S11(lambda_k)|^2`` across the target wavelengths [dimensionless].
    t_lower_bound, t_upper_bound : float, optional
        Linear power bounds for transmission (default ``10^(-0.5/10)`` to ``1.0``).
    r_lower_bound, r_upper_bound : float, optional
        Linear power bounds for reflection (default ``0.0`` to ``10^(-20/10)``).

    Returns
    -------
    float
        Scalar evaluation metric (positive when all channels lie inside both target windows).
    """
    s21 = np.asarray(s21_sq, dtype=np.float64)
    s11 = np.asarray(s11_sq, dtype=np.float64)
    margin_t = (s21 - float(t_lower_bound)) / (float(t_upper_bound) - float(t_lower_bound))
    margin_r = (float(r_upper_bound) - s11) / (float(r_upper_bound) - float(r_lower_bound))
    return float(min(np.min(margin_t), np.min(margin_r)))


# ---------------------------------------------------------------------------
# Wakis mode-port helpers from the invrs-wakis bridge (.ref/invrs_wakis)
# ---------------------------------------------------------------------------


def _z_fraction(mode) -> float:
    """Fraction of transverse |E|^2 carried by the Ez component of a waveguide mode."""
    axis = {"b": mode.axes[1], "c": mode.axes[2]}
    ez = sum(np.sum(np.abs(v) ** 2) for k, v in mode.e.items() if axis[k] == "z")
    total = sum(np.sum(np.abs(v) ** 2) for v in mode.e.values())
    return float(ez / max(total, 1e-30))


def polarized_modes(solver, frequency: float, center, size, n_modes: int = 1, polarization: str = "Ez"):
    """Guided modes of one polarization family ('Ez' TM or 'Hz' TE), sorted by decreasing neff."""
    want_ez = polarization == "Ez"
    modes = solve_modes(solver, frequency, center, size, n_modes=2 * n_modes + 2)
    picked = [m for m in modes if (_z_fraction(m) > 0.5) == want_ez]
    return picked[:n_modes]


class PolarizedModeSource(ModeSource):
    """Wakis ``ModeSource`` restricted to a single 2D polarization family ('Ez')."""

    def __init__(self, *args, polarization: str = "Ez", **kwargs):
        super().__init__(*args, **kwargs)
        self.polarization = polarization
        self.family_index = self.mode_index

    def _initialize(self, solver):
        n = self.family_index + 1
        family = polarized_modes(
            solver, self.mode_frequency, self.center, self.size, n, self.polarization
        )
        if len(family) < n:
            raise ValueError(f"{self.polarization} mode {n} does not propagate.")
        target = family[self.family_index]
        modes = solve_modes(
            solver, self.mode_frequency, self.center, self.size, n_modes=2 * n + 2
        )
        self.mode_index = int(np.argmin([abs(m.beta - target.beta) for m in modes]))
        super()._initialize(solver)


class PolarizedModeMonitor(ModeMonitor):
    """Wakis ``ModeMonitor`` restricted to a single 2D polarization family ('Ez')."""

    def __init__(self, *args, polarization: str = "Ez", **kwargs):
        super().__init__(*args, **kwargs)
        self.polarization = polarization

    def _solve_modes(self):
        def family(f):
            return polarized_modes(
                self.solver, f, self.center, self.size, self.n_modes, self.polarization
            )

        if self.mode_frequency is not None:
            self.modes = [family(self.mode_frequency)] * self.frequencies.size
        else:
            self.modes = [family(f) for f in self.frequencies]


def load_round2_artifacts() -> dict:
    """Load the committed Round-2 waveguide bend artifacts from ``data/round2_bend/``."""
    design = np.load(ROUND2_DIR / "design.npy")
    fc = np.load(ROUND2_DIR / "fields_compact.npz")
    snaps = np.load(ROUND2_DIR / "snapshots_thinned.npz")
    summary = json.loads((ROUND2_DIR / "summary.json").read_text())
    log_records = [
        json.loads(line)
        for line in (ROUND2_DIR / "log.jsonl").read_text().splitlines()
        if line.strip()
    ]
    eps_full = fc["eps"].astype(np.float64) * float(fc["eps_scale"])
    ez_full = (
        fc["Ez_re"].astype(np.float64) + 1j * fc["Ez_im"].astype(np.float64)
    ) * float(fc["Ez_scale"])
    dense_s = (
        fc["dense_S_re"].astype(np.float64) + 1j * fc["dense_S_im"].astype(np.float64)
    ) * float(fc["dense_S_scale"])
    return {
        "design": design,
        "fields_compact": fc,
        "eps_full": eps_full,
        "ez_full": ez_full,
        "dense_S": dense_s,
        "dense_wavelengths_nm": fc["dense_wavelengths_nm"].astype(np.float64),
        "wavelengths_nm": fc["wavelengths_nm"].astype(np.float64),
        "x_um": fc["x_um"].astype(np.float64),
        "y_um": fc["y_um"].astype(np.float64),
        "ports": fc["ports"].astype(np.float64),
        "design_um": fc["design_um"].astype(np.float64),
        "pml_um": float(fc["pml_um"]),
        "snapshots": snaps,
        "summary": summary,
        "log": log_records,
    }

---

## Section 1 · Round-2 Optimization Trace & Imageruler Length-Scale Verification (`task_0`)

We load our completed Round-2 `ceviche_waveguide_bend` optimization from
`data/round2_bend/` (`design.npy`, `log.jsonl`, `snapshots_thinned.npz`,
`summary.json`, and `fields_compact.npz`) and verify:
1. The 145-step optimization trajectory across the four Stage-1 L-BFGS-B
   $\beta$-continuation stages ($\beta = 4, 8, 16, 32$) and the 40-step Stage-2
   Adam STE stage ($\beta = 64$).
2. The `imageruler`-style minimum solid width and void spacing on the final
   $160\times 160$ binary design (`data/round2_bend/design.npy` embedded in its
   waveguide surroundings from `fields_compact.npz`), confirming zero interior
   violations for all circular brush diameters $d \le 8\text{ px}$ ($80\text{ nm}$)
   and first violations at $d = 9\text{ px}$.

In [ ]:
def run_task_0_round2_trace_and_length_scale(cfg: dict) -> dict:
    """Run Task 0: verify Round-2 optimization trace and 8/8 px length scale on design.npy."""
    art = load_round2_artifacts()
    design = art["design"]
    eps_full = art["eps_full"]
    snaps = art["snapshots"]
    summ = art["summary"]
    log_recs = art["log"]

    # Extract surrounding window [92-border : 252+border] from the 313x313 permittivity map
    border = 12
    x0, y0, x1, y1 = 92, 92, 252, 252
    win_binary = (eps_full[x0 - border : x1 + border, y0 - border : y1 + border] > 7.0).astype(bool)

    diameters = np.arange(1, 12, dtype=np.int32)
    solid_violations = np.zeros_like(diameters)
    void_violations = np.zeros_like(diameters)
    for idx_d, d_val in enumerate(diameters):
        vs, vv = count_interior_violations(
            design, int(d_val), surrounding_window=win_binary, border=border
        )
        solid_violations[idx_d] = vs
        void_violations[idx_d] = vv

    mw_design, ms_design = measure_length_scales_px(
        design, max_px=11, surrounding_window=win_binary, border=border
    )

    # Measure length scale and binarization across all 50 snapshots in snapshots_thinned.npz
    snap_designs = snaps["designs"].astype(np.float64)
    n_snaps = len(snap_designs)
    snap_binarization = np.zeros(n_snaps, dtype=np.float64)
    snap_min_width = np.zeros(n_snaps, dtype=np.int32)
    snap_min_space = np.zeros(n_snaps, dtype=np.int32)
    for k in range(n_snaps):
        d_k = snap_designs[k]
        snap_binarization[k] = float(1.0 - np.mean(4.0 * d_k * (1.0 - d_k)))
        w_k, s_k = measure_length_scales_px(d_k, max_px=10)
        snap_min_width[k] = w_k
        snap_min_space[k] = s_k

    # Parse per-iteration trace and stage-end repaired evaluations from log.jsonl
    iter_rows = [r for r in log_recs if "step" in r]
    repaired_rows = [r for r in log_recs if "repaired_eval_metric" in r]
    log_loss = np.array([float(r["loss"]) for r in iter_rows], dtype=np.float64)
    log_eval = np.array([float(r["eval_metric"]) for r in iter_rows], dtype=np.float64)
    log_stage = np.array([str(r["stage"]) for r in iter_rows])
    log_step = np.array([int(r["step"]) for r in iter_rows], dtype=np.int32)
    log_time = np.array([float(r["time"]) for r in iter_rows], dtype=np.float64)
    repaired_stages = np.array([str(r["stage"]) for r in repaired_rows])
    repaired_evals = np.array([float(r["repaired_eval_metric"]) for r in repaired_rows], dtype=np.float64)

    # Synthetic brush repair demonstration on a defective test pattern
    n_demo = int(cfg["demo_n_design"])
    d_demo = int(cfg["demo_target_px"])
    ax = np.linspace(-1.0, 1.0, n_demo)
    xx, yy = np.meshgrid(ax, ax, indexing="ij")
    r_arc = np.hypot(xx + 0.6, yy - 0.6)
    b_unrepaired = (np.exp(-8.0 * (r_arc - 0.85) ** 2) >= 0.45).astype(np.float64)
    b_unrepaired[n_demo // 2, :] = 1.0  # 1-px solid bridge
    b_unrepaired[n_demo // 3, n_demo // 3] = 0.0  # 1-px void pinhole
    w_before, s_before = measure_length_scales_px(b_unrepaired, max_px=d_demo + 3)
    b_repaired = morphological_repair(b_unrepaired, diameter_px=d_demo)
    w_after, s_after = measure_length_scales_px(b_repaired, max_px=d_demo + 3)

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        DATA_DIR / "length_scale_check.npz",
        diameters=diameters,
        solid_violations=solid_violations,
        void_violations=void_violations,
        measured_solid_void_px=np.array([mw_design, ms_design], dtype=np.int32),
        stored_leaderboard_px=np.array(summ["length_scale_leaderboard_px"], dtype=np.int32),
        stored_solid_void_px=np.array(summ["length_scale_solid_void_px"], dtype=np.int32),
        snap_kept_index=snaps["kept_index"].astype(np.int32),
        snap_stage=snaps["stage"],
        snap_eval_metric=snaps["eval_metric"].astype(np.float64),
        snap_loss=snaps["loss"].astype(np.float64),
        snap_binarization=snap_binarization,
        snap_min_width=snap_min_width,
        snap_min_space=snap_min_space,
        log_loss=log_loss,
        log_eval=log_eval,
        log_stage=log_stage,
        log_step=log_step,
        log_time=log_time,
        repaired_stages=repaired_stages,
        repaired_evals=repaired_evals,
        b_unrepaired=b_unrepaired.astype(np.float16),
        b_repaired=b_repaired.astype(np.float16),
        demo_target_px=np.array(d_demo, dtype=np.int32),
        demo_before_px=np.array([w_before, s_before], dtype=np.int32),
        demo_after_px=np.array([w_after, s_after], dtype=np.int32),
    )
    return {
        "round2_design_shape": list(design.shape),
        "measured_length_scale_px": [mw_design, ms_design],
        "stored_length_scale_px": list(summ["length_scale_leaderboard_px"]),
        "n_logged_iterations": len(log_loss),
        "best_ste_wakis_eval_metric": float(np.max(log_eval)),
        "official_eval_metric": float(summ["official_eval_metric"]),
    }

---

## Section 2 · Broadband Wakis Re-Simulation of Our Round-2 Design (`task_1`)

Since `invrs_gym` and `ceviche_challenges` are not required at runtime, we
rebuild the `ceviche_waveguide_bend` simulation directly from the geometry metadata
stored in `data/round2_bend/fields_compact.npz` and `data/round2_bend/design.npy`,
following `CevicheGeometry` and `CevicheSimulation` in `.ref/invrs_wakis/ceviche.py`:
- Full domain: $312 \times 312$ pixels at $\Delta l = 10\text{ nm}$ ($3.12\,\mu\text{m} \times 3.12\,\mu\text{m}$),
  with $200\text{ nm}$ CPML on $\pm x, \pm y$ and one periodic cell along $z$.
- Design region $[0.92, 2.52]\,\mu\text{m} \times [0.92, 2.52]\,\mu\text{m}$ ($160\times 160$ pixels)
  populated with our binary `round2_bend/design.npy` ($\varepsilon_{\text{clad}} = 2.25$,
  $\varepsilon_{\text{slab}} = 12.25$).
- Input `PolarizedModeSource` at Port 1 ($x = 0.24501\,\mu\text{m}, y = 1.72\,\mu\text{m}$,
  width $1.44\,\mu\text{m}$) and `PolarizedModeMonitor` readouts at Port 1 and Port 2
  ($x = 1.72\,\mu\text{m}, y = 0.24501\,\mu\text{m}$, offset $50\text{ nm}$ into the device)
  sampled at the pre-warped frequencies $f_{\text{sim}}$ (`match_fdfd=True`) across all
  67 dense wavelengths ($1245\text{--}1315\text{ nm}$).

In [ ]:
def resimulate_round2_bend_wakis(
    stride: int = 2,
    fwidth: float = 0.12,
    run_time: float = 1.8e-13,
) -> dict:
    """Re-simulate the stored Round-2 waveguide bend design in Wakis and return spectra."""
    art = load_round2_artifacts()
    design = art["design"]
    eps_full = art["eps_full"].copy()
    # Explicitly inject design.npy into the [92:252, 92:252] design region
    eps_nodes_10nm = np.where(eps_full > 7.0, 12.25, 2.25)
    eps_nodes_10nm[92:252, 92:252] = 2.25 + (12.25 - 2.25) * np.asarray(design, dtype=np.float64)

    stride = int(stride)
    dl = 10e-9 * stride
    nx = 312 // stride
    ny = 312 // stride
    n_pml = 20 // stride

    if stride == 1:
        eps_nodes = eps_nodes_10nm
    else:
        pix = eps_nodes_10nm[:312, :312].reshape(nx, stride, ny, stride).mean(axis=(1, 3))
        ix = np.minimum(np.arange(nx + 1), nx - 1)
        eps_nodes = pix[np.ix_(ix, ix)]

    dt_2d = dl / (C0 * np.sqrt(2.0))
    dt = 0.95 * dt_2d

    dense_wl_nm = art["dense_wavelengths_nm"]
    wl_6_nm = art["wavelengths_nm"]
    f_target = C0 / (dense_wl_nm * 1e-9)
    # Pre-warp frequencies to remove leapfrog temporal dispersion (match_fdfd=True)
    f_sim = np.arcsin(np.pi * f_target * dt) / (np.pi * dt)
    fc_pulse = 0.5 * (float(f_sim.min()) + float(f_sim.max()))
    pulse = GaussianSource(fc_pulse, float(fwidth) * fc_pulse)
    Nt = int(np.ceil((pulse.t0 + float(run_time)) / dt))

    grid = wakis.GridFIT3D(
        dl / 2,
        nx * dl + dl / 2,
        dl / 2,
        ny * dl + dl / 2,
        -dl / 2,
        dl / 2,
        nx,
        ny,
        1,
        verbose=0,
    )
    solver = wakis.SolverFIT3D(
        grid,
        bc_low=["cpml", "cpml", "periodic"],
        bc_high=["cpml", "cpml", "periodic"],
        n_pml=n_pml,
        dt=dt,
        verbose=0,
    )
    for comp in "xyz":
        solver.ieps[:, :, 0, comp] = solver.ieps[:, :, 0, comp] / eps_nodes

    p0, p1 = art["ports"][0], art["ports"][1]
    offset_um = 0.05  # 5 pixels (50 nm) into the device
    src = PolarizedModeSource(
        pulse,
        center=(p0[0] * 1e-6, p0[1] * 1e-6, 0.0),
        size=(0.0, p0[2] * 1e-6, np.inf),
        direction="+",
        mode_index=0,
    )
    m0 = PolarizedModeMonitor(
        f_sim,
        center=((p0[0] + offset_um) * 1e-6, p0[1] * 1e-6, 0.0),
        size=(0.0, p0[2] * 1e-6, np.inf),
        n_modes=1,
        mode_frequency=fc_pulse,
    )
    m1 = PolarizedModeMonitor(
        f_sim,
        center=(p1[0] * 1e-6, (p1[1] + offset_um) * 1e-6, 0.0),
        size=(p1[2] * 1e-6, 0.0, np.inf),
        n_modes=1,
        mode_frequency=fc_pulse,
    )

    for step in range(Nt):
        t = step * dt
        src.update(solver, t)
        solver.one_step()
        m0.update(solver, t)
        m1.update(solver, t)

    a0_plus, a0_minus = m0.get_amplitudes(0)
    a1_plus, a1_minus = m1.get_amplitudes(0)
    # Port 0 is excited in +x; Port 1 points +y into the device so outgoing wave is along -y (a1_minus)
    s11_resim = a0_minus / a0_plus
    s21_resim = a1_minus / a0_plus
    s11_sq_resim = np.abs(s11_resim) ** 2
    s21_sq_resim = np.abs(s21_resim) ** 2

    dense_s_stored = art["dense_S"]
    s11_sq_stored = np.abs(dense_s_stored[:, 0]) ** 2
    s21_sq_stored = np.abs(dense_s_stored[:, 1]) ** 2

    idx_6 = np.array([int(np.argmin(np.abs(dense_wl_nm - w))) for w in wl_6_nm], dtype=np.int32)
    s21_6_resim = s21_sq_resim[idx_6]
    s11_6_resim = s11_sq_resim[idx_6]
    s21_6_stored_dense = s21_sq_stored[idx_6]
    s11_6_stored_dense = s11_sq_stored[idx_6]

    eval_resim = compute_bend_eval_metric(s21_6_resim, s11_6_resim)
    eval_stored_dense = compute_bend_eval_metric(s21_6_stored_dense, s11_6_stored_dense)

    return {
        "stride": stride,
        "dl_nm": dl * 1e9,
        "nx": nx,
        "ny": ny,
        "Nt": Nt,
        "dense_wavelengths_nm": dense_wl_nm,
        "wavelengths_6_nm": wl_6_nm,
        "idx_6": idx_6,
        "s11_sq_resim": s11_sq_resim,
        "s21_sq_resim": s21_sq_resim,
        "s11_sq_stored": s11_sq_stored,
        "s21_sq_stored": s21_sq_stored,
        "s11_6_resim": s11_6_resim,
        "s21_6_resim": s21_6_resim,
        "eval_metric_resim": eval_resim,
        "eval_metric_stored_dense": eval_stored_dense,
        "max_abs_diff_s21_dense": float(np.max(np.abs(s21_sq_resim - s21_sq_stored))),
        "mean_abs_diff_s21_dense": float(np.mean(np.abs(s21_sq_resim - s21_sq_stored))),
        "max_abs_diff_s11_dense": float(np.max(np.abs(s11_sq_resim - s11_sq_stored))),
    }


def run_task_1_wakis_resimulation(cfg: dict) -> dict:
    """Run Task 1: re-simulate our Round-2 design in Wakis and compare with stored dense_S."""
    res = resimulate_round2_bend_wakis(
        stride=int(cfg["stride"]),
        fwidth=float(cfg["fwidth"]),
        run_time=float(cfg["run_time"]),
    )
    art = load_round2_artifacts()
    summ = art["summary"]
    s2_off = np.array(summ["official_S2"], dtype=np.float64)[:, 0, :]
    s2_wakis_cev = np.array(summ["wakis_r1_ceviche_estimator_S2"], dtype=np.float64)[:, 0, :]
    s2_wakis_mod = np.array(summ["wakis_r1_modal_S2"], dtype=np.float64)[:, 0, :]

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        DATA_DIR / "wakis_resimulation.npz",
        stride=np.array(res["stride"], dtype=np.int32),
        dl_nm=np.array(res["dl_nm"], dtype=np.float64),
        nx=np.array(res["nx"], dtype=np.int32),
        ny=np.array(res["ny"], dtype=np.int32),
        Nt=np.array(res["Nt"], dtype=np.int32),
        dense_wavelengths_nm=res["dense_wavelengths_nm"],
        wavelengths_6_nm=res["wavelengths_6_nm"],
        idx_6=res["idx_6"],
        s11_sq_resim=res["s11_sq_resim"],
        s21_sq_resim=res["s21_sq_resim"],
        s11_sq_stored=res["s11_sq_stored"],
        s21_sq_stored=res["s21_sq_stored"],
        s11_6_resim=res["s11_6_resim"],
        s21_6_resim=res["s21_6_resim"],
        s2_official_6=s2_off,
        s2_wakis_cev_6=s2_wakis_cev,
        s2_wakis_mod_6=s2_wakis_mod,
        eval_metric_resim=np.array(res["eval_metric_resim"], dtype=np.float64),
        eval_metric_stored_dense=np.array(res["eval_metric_stored_dense"], dtype=np.float64),
        max_abs_diff_s21_dense=np.array(res["max_abs_diff_s21_dense"], dtype=np.float64),
        mean_abs_diff_s21_dense=np.array(res["mean_abs_diff_s21_dense"], dtype=np.float64),
        max_abs_diff_s11_dense=np.array(res["max_abs_diff_s11_dense"], dtype=np.float64),
    )
    return {
        "stride": res["stride"],
        "grid_shape": [res["nx"], res["ny"]],
        "dl_nm": res["dl_nm"],
        "Nt": res["Nt"],
        "eval_metric_resim": round(res["eval_metric_resim"], 4),
        "max_abs_diff_s21_dense": round(res["max_abs_diff_s21_dense"], 5),
        "mean_abs_diff_s21_dense": round(res["mean_abs_diff_s21_dense"], 5),
        "max_abs_diff_s11_dense": round(res["max_abs_diff_s11_dense"], 5),
    }

---

## Section 3 · Leaderboard & Round-1/Round-2 Benchmark Comparison (`task_2`)

We compile the official `invrs-gym` scores, length scales, and S-parameter
spectra from `data/round2_bend/summary.json`, `data/leaderboard_rescore.json`,
and `data/round1/*waveguide_bend*.json`, together with the memory-ladder
projections for the $313\times 313$ grid from `studies/adjoint-memory/projection.json`.

In [ ]:
def run_task_2_leaderboard_and_round1_comparison(cfg: dict) -> dict:
    """Run Task 2: compile verified Round-2, Round-1, and leaderboard benchmark data."""
    art = load_round2_artifacts()
    summ_r2 = art["summary"]
    lb_entries = json.loads(LEADERBOARD_PATH.read_text())
    lb_bend = [e for e in lb_entries if e["challenge"] == "ceviche_waveguide_bend"]
    rahul_entry = next(e for e in lb_bend if "rahulkpadhy" in e["file"])
    ours_seed_a = next(e for e in lb_bend if "ours_full_bend_seed1" in e["file"])

    r1_seed0 = json.loads(
        (
            ROUND1_DIR
            / "20261018-0430-invrs-ceviche__shard_0__ceviche_lightweight_waveguide_bend_seed0__summary.json"
        ).read_text()
    )
    r1_seed1 = json.loads(
        (
            ROUND1_DIR
            / "20261018-0430-invrs-ceviche__shard_1__ceviche_lightweight_waveguide_bend_seed1__summary.json"
        ).read_text()
    )

    s2_r2_off = np.array(summ_r2["official_S2"], dtype=np.float64)[:, 0, :]
    s2_r2_cev = np.array(summ_r2["wakis_r1_ceviche_estimator_S2"], dtype=np.float64)[:, 0, :]
    s2_r2_mod = np.array(summ_r2["wakis_r1_modal_S2"], dtype=np.float64)[:, 0, :]
    s2_seed_a = np.array(ours_seed_a["s2"], dtype=np.float64)[:, 0, :]
    s2_rahul = np.array(rahul_entry["s2"], dtype=np.float64)[:, 0, :]

    r1_s0_off = np.array(r1_seed0["official_S2"], dtype=np.float64)[:, 0, :]
    r1_s0_cev = np.array(r1_seed0["wakis_r1_ceviche_estimator_S2"], dtype=np.float64)[:, 0, :]
    r1_s1_off = np.array(r1_seed1["official_S2"], dtype=np.float64)[:, 0, :]
    r1_s1_cev = np.array(r1_seed1["wakis_r1_ceviche_estimator_S2"], dtype=np.float64)[:, 0, :]

    proj_path = STUDY_DIR / "projection.json"
    bend_memory_proj = {}
    if proj_path.exists():
        all_proj = json.loads(proj_path.read_text())
        bend_memory_proj = all_proj.get("ceviche_waveguide_bend", {})

    compiled = {
        "challenge": "ceviche_waveguide_bend",
        "grid_spacing_nm": 10.0,
        "domain_pixels": [312, 312],
        "ez_nodes": [313, 313],
        "design_region_pixels": [160, 160],
        "wavelengths_nm": CHALLENGE_WAVELENGTHS_NM.tolist(),
        "target_length_scale_px": [8, 8],
        "leaderboard_rescore": {
            "wakis_round2_best": {
                "source": "data/round2_bend/summary.json (ste_beta64:34)",
                "eval_metric": float(summ_r2["official_eval_metric"]),
                "wakis_r1_modal_eval_metric": float(summ_r2["wakis_r1_modal_eval_metric"]),
                "wakis_r1_ceviche_eval_metric": float(summ_r2["wakis_r1_ceviche_estimator_eval_metric"]),
                "official_loss": float(summ_r2["official_loss"]),
                "min_width_px": int(summ_r2["length_scale_leaderboard_px"][0]),
                "min_spacing_px": int(summ_r2["length_scale_leaderboard_px"][1]),
                "feasible_8px": True,
                "min_s21_db": float(10.0 * np.log10(np.min(s2_r2_off[:, 1]))),
                "max_s21_db": float(10.0 * np.log10(np.max(s2_r2_off[:, 1]))),
                "max_s11_db": float(10.0 * np.log10(np.max(s2_r2_off[:, 0]))),
                "s2": s2_r2_off.tolist(),
            },
            "wakis_round2_seed_a": {
                "source": "data/leaderboard_rescore.json (ours_full_bend_seed1.npy)",
                "eval_metric": float(ours_seed_a["eval_metric"]),
                "min_width_px": int(ours_seed_a["minimum_width"]),
                "min_spacing_px": int(ours_seed_a["minimum_spacing"]),
                "feasible_8px": True,
                "min_s21_db": float(10.0 * np.log10(np.min(s2_seed_a[:, 1]))),
                "max_s21_db": float(10.0 * np.log10(np.max(s2_seed_a[:, 1]))),
                "max_s11_db": float(10.0 * np.log10(np.max(s2_seed_a[:, 0]))),
                "s2": s2_seed_a.tolist(),
            },
            "invrs_gym_leaderboard_best": {
                "source": f"data/leaderboard_rescore.json ({rahul_entry['file']})",
                "eval_metric": float(rahul_entry["eval_metric"]),
                "published": float(rahul_entry["published"]),
                "min_width_px": int(rahul_entry["minimum_width"]),
                "min_spacing_px": int(rahul_entry["minimum_spacing"]),
                "feasible_8px": False,
                "min_s21_db": float(10.0 * np.log10(np.min(s2_rahul[:, 1]))),
                "max_s21_db": float(10.0 * np.log10(np.max(s2_rahul[:, 1]))),
                "max_s11_db": float(10.0 * np.log10(np.max(s2_rahul[:, 0]))),
                "s2": s2_rahul.tolist(),
            },
            "wakis_round1_seed1": {
                "source": "data/round1/...ceviche_lightweight_waveguide_bend_seed1__summary.json",
                "eval_metric_40nm": float(r1_seed1["official_eval_metric"]),
                "eval_metric_10nm": float(r1_seed1["ceviche_10nm_eval_metric"]),
                "min_width_40nm_px": int(r1_seed1["length_scale_solid_void_px"][0]),
                "min_spacing_40nm_px": int(r1_seed1["length_scale_solid_void_px"][1]),
                "target_40nm_px": list(r1_seed1["target_min_width_spacing_px"]),
                "feasible": False,
            },
            "wakis_round1_seed0": {
                "source": "data/round1/...ceviche_lightweight_waveguide_bend_seed0__summary.json",
                "eval_metric_40nm": float(r1_seed0["official_eval_metric"]),
                "eval_metric_10nm": float(r1_seed0["ceviche_10nm_eval_metric"]),
                "min_width_40nm_px": int(r1_seed0["length_scale_solid_void_px"][0]),
                "min_spacing_40nm_px": int(r1_seed0["length_scale_solid_void_px"][1]),
                "target_40nm_px": list(r1_seed0["target_min_width_spacing_px"]),
                "feasible": True,
            },
        },
        "claims_c55_spread": {
            "seed_scores": [float(ours_seed_a["eval_metric"]), float(summ_r2["official_eval_metric"])],
            "spread": round(float(summ_r2["official_eval_metric"]) - float(ours_seed_a["eval_metric"]), 4),
            "mean": round(
                0.5 * (float(ours_seed_a["eval_metric"]) + float(summ_r2["official_eval_metric"])), 4
            ),
        },
        "cross_validation_wakis_vs_ceviche_fdfd": {
            "round2_10nm_max_abs_delta_s_sq_cev_est": float(np.max(np.abs(s2_r2_off - s2_r2_cev))),
            "round2_10nm_mean_abs_delta_s_sq_cev_est": float(np.mean(np.abs(s2_r2_off - s2_r2_cev))),
            "round2_10nm_max_abs_delta_s_sq_modal": float(np.max(np.abs(s2_r2_off - s2_r2_mod))),
            "round1_40nm_seed0_max_abs_delta_s_sq": float(np.max(np.abs(r1_s0_off - r1_s0_cev))),
            "round1_40nm_seed1_max_abs_delta_s_sq": float(np.max(np.abs(r1_s1_off - r1_s1_cev))),
            "ceviche_reflection_floor_db": -31.0,
        },
        "candidates_round2": summ_r2["candidates"],
        "preflight_round2": summ_r2["preflight"],
        "memory_projection_313x313_56587steps": bend_memory_proj,
    }

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with open(DATA_DIR / "round2_evaluation_summary.json", "w") as f:
        json.dump(compiled, f, indent=2)
    return {
        "wakis_round2_best": round(float(summ_r2["official_eval_metric"]), 4),
        "wakis_round2_seed_a": round(float(ours_seed_a["eval_metric"]), 4),
        "leaderboard_best": round(float(rahul_entry["eval_metric"]), 4),
        "round2_max_fdfd_diff": float(np.max(np.abs(s2_r2_off - s2_r2_cev))),
    }

---

## Section 4 · Toy Problem (Not the Challenge): Live Two-Stage Adjoint + STE Demo (`task_3`)

> **Note — Toy problem (not the challenge)**: The full $312\times 312$-pixel,
> 145-iteration `ceviche_waveguide_bend` optimization took $979\text{ s}$ on an L4 GPU
> (`data/round2_bend/summary.json`). To let readers step through the exact two-stage
> density ($\beta$-continuation) + morphological brush STE loop interactively in a
> few seconds on a laptop CPU, `task_3` runs a miniature $40\times 40$-cell toy
> bend using `wakis.inverse.AdjointSimulation` and saves its output exclusively to
> `data/toy_bend_demo.npz` with the label `"toy problem (not the challenge)"`.

In [ ]:
def conic_kernel_2d(radius_cells: float) -> np.ndarray:
    r = int(np.ceil(radius_cells))
    if r <= 0:
        return np.ones((1, 1), dtype=np.float64)
    ax = np.arange(-r, r + 1, dtype=np.float64)
    xx, yy = np.meshgrid(ax, ax, indexing="ij")
    k = np.maximum(0.0, 1.0 - np.hypot(xx, yy) / float(radius_cells))
    return k / float(k.sum())


def conic_filter_np(
    rho: np.ndarray, radius_cells: float
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    k = conic_kernel_2d(radius_cells)
    r2 = np.asarray(rho[:, :, 0] if rho.ndim == 3 else rho, dtype=np.float64)
    den = convolve2d(np.ones_like(r2), k, mode="same")
    out2 = convolve2d(r2, k, mode="same") / den
    out = out2[:, :, None] if rho.ndim == 3 else out2
    return out, k, den


def conic_filter_vjp_np(g_tilde: np.ndarray, k: np.ndarray, den: np.ndarray) -> np.ndarray:
    g2 = np.asarray(g_tilde[:, :, 0] if g_tilde.ndim == 3 else g_tilde, dtype=np.float64)
    g_raw2 = convolve2d(g2 / den, k, mode="same")
    return g_raw2[:, :, None] if g_tilde.ndim == 3 else g_raw2


def tanh_projection_np(
    rho_tilde: np.ndarray, beta: float = 8.0, eta: float = 0.5
) -> tuple[np.ndarray, np.ndarray]:
    b = float(beta)
    e = float(eta)
    th_be = np.tanh(b * e)
    th_b1 = np.tanh(b * (1.0 - e))
    denom = th_be + th_b1
    u = np.tanh(b * (rho_tilde - e))
    rho_bar = (th_be + u) / denom
    dbar_dtilde = b * (1.0 - u**2) / denom
    return rho_bar, dbar_dtilde


def run_task_3_toy_bend_teaching_optimization(cfg: dict) -> dict:
    """Run Task 3: small toy problem (not the challenge) demonstrating Stage-1 + Stage-2 STE in Wakis."""
    n_grid = int(cfg["n_grid"])
    n_design = int(cfg["n_design"])
    Nt = int(cfg["Nt"])
    n1 = int(cfg["n_iters_stage1"])
    n2 = int(cfg["n_iters_stage2"])
    brush_px = int(cfg["brush_px"])
    n_total = n1 + n2

    f0_toy = 3.0e9
    lam0_toy = C0 / f0_toy
    dx = lam0_toy / 28.0
    dz = 10.0 * dx
    L = n_grid * dx
    w = 6.0 * dx
    D = n_design * dx
    eps_wg = 6.0
    n_pml = 8
    freqs_toy = f0_toy * np.linspace(0.98, 1.02, 3)

    def make_toy_solver():
        grid = wakis.GridFIT3D(
            -L / 2, L / 2, -L / 2, L / 2, -dz / 2, dz / 2, n_grid, n_grid, 1, verbose=0
        )
        solver = wakis.SolverFIT3D(
            grid,
            bc_low=["cpml", "cpml", "periodic"],
            bc_high=["cpml", "cpml", "periodic"],
            n_pml=n_pml,
            verbose=0,
        )
        for center, size in (
            ((-L / 4, 0.0, 0.0), (L / 2, w, 0.0)),
            ((0.0, L / 4, 0.0), (w, L / 2, 0.0)),
        ):
            guide = MaterialGrid(center, size, (1, 1, 1), eps_min=eps_wg, eps_max=eps_wg)
            guide.apply(solver, np.ones((1, 1, 1)))
        return solver

    design_grid = MaterialGrid(
        (0.0, 0.0, 0.0), (D, D, 0.0), (n_design, n_design, 1), eps_min=1.0, eps_max=eps_wg
    )
    x_src = -L / 2 + (n_pml + 3) * dx
    y_out = L / 2 - (n_pml + 3) * dx
    src = CurrentSource(
        GaussianSource(f0_toy, 0.65 * f0_toy),
        component="z",
        center=(x_src, 0.0, 0.0),
        size=(0.0, w, 0.0),
    )
    monitors = {
        "out_port": wakis.DFTMonitor(
            freqs_toy, components=["Ez"], center=(0.0, y_out, 0.0), size=(w, 0.0, 0.0)
        )
    }
    sim = AdjointSimulation(make_toy_solver, [src], monitors, design_grid, Nt)

    radius_cells = float(brush_px) * 0.85
    k_filt = conic_kernel_2d(radius_cells)
    den_filt = convolve2d(np.ones((n_design, n_design)), k_filt, mode="same")

    q = max(1, n1 // 2)
    beta_schedule = np.array([4.0] * q + [16.0] * (n1 - q) + [64.0] * n2, dtype=np.float64)
    is_ste_stage = np.array([False] * n1 + [True] * n2, dtype=bool)

    rho_raw = np.full(sim.design.shape, 0.5, dtype=np.float64)
    obj_trace = np.zeros(n_total, dtype=np.float64)
    for it in range(n_total):
        beta_val = float(beta_schedule[it])
        ste_active = bool(is_ste_stage[it])
        rho_tilde, _, _ = conic_filter_np(rho_raw, radius_cells)
        rho_proj, dproj_dtilde = tanh_projection_np(rho_tilde, beta=beta_val, eta=0.5)
        if ste_active:
            rho_eval = morphological_repair(
                (rho_tilde >= 0.5).astype(np.float64), diameter_px=brush_px
            )
            _, dsurr = tanh_projection_np(rho_tilde, beta=16.0, eta=0.5)
            dbar_dtilde = 0.5 * dproj_dtilde + 0.5 * dsurr
        else:
            rho_eval = rho_proj
            dbar_dtilde = dproj_dtilde

        out = sim.forward(rho_eval)
        ez_out = out["out_port"]["Ez"]
        obj_val = float(np.mean(np.sum(np.abs(ez_out) ** 2, axis=(1, 2, 3))))
        obj_trace[it] = obj_val

        cot_out = (2.0 / len(freqs_toy)) * ez_out
        g_bar = sim.vjp({"out_port": {"Ez": cot_out}})
        g_tilde = g_bar * dbar_dtilde
        g_raw = conic_filter_vjp_np(g_tilde, k_filt, den_filt)
        g_unit = g_raw / (float(np.max(np.abs(g_raw))) + 1e-30)
        lr = 0.16 if not ste_active else 0.08
        rho_raw = np.clip(rho_raw + lr * g_unit, 0.0, 1.0)

    rho_tilde_fin, _, _ = conic_filter_np(rho_raw, radius_cells)
    toy_binary = morphological_repair(
        (rho_tilde_fin >= 0.5).astype(np.float64), diameter_px=brush_px
    )[:, :, 0]
    mw_toy, ms_toy = measure_length_scales_px(toy_binary, max_px=brush_px + 2)

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        DATA_DIR / "toy_bend_demo.npz",
        label=np.array("toy problem (not the challenge)"),
        toy_binary=toy_binary.astype(np.float16),
        obj_trace=obj_trace,
        beta_schedule=beta_schedule,
        is_ste_stage=is_ste_stage,
        brush_px=np.array(brush_px, dtype=np.int32),
        measured_px=np.array([mw_toy, ms_toy], dtype=np.int32),
    )
    return {
        "label": "toy problem (not the challenge)",
        "n_grid": n_grid,
        "n_design": n_design,
        "obj_improvement_ratio": round(float(obj_trace[-1] / max(obj_trace[0], 1e-30)), 3),
        "toy_length_scale_px": [mw_toy, ms_toy],
    }


def run_task_4_optional_invrs_gym(cfg: dict) -> dict:
    """Run Task 4: optional official invrs-gym re-score of data/round2_bend/design.npy if installed."""
    try:
        import invrs_gym  # type: ignore[import-not-found]
    except ImportError:
        res = {
            "available": False,
            "reason": (
                "invrs_gym is not installed in the local offline environment. "
                "Install with `pip install invrs-wakis invrs-gym` to run official gym scoring; "
                "using verified stored score +0.6427 from data/round2_bend/summary.json."
            ),
            "stored_official_eval_metric": 0.6427273992133412,
        }
    else:
        res = {
            "available": True,
            "invrs_gym_version": getattr(invrs_gym, "__version__", "unknown"),
            "stored_official_eval_metric": 0.6427273992133412,
        }
    return res

---

## Section 5 · Visualization & Figure Generation

Calling `plot_all()` (or passing `--plot`) renders eight publication-quality
figures in `figures/` directly from `data/round2_bend/`, `data/leaderboard_rescore.json`,
`data/round1/*.json`, and the Wakis re-simulation outputs in `data/`:

1. `01_bend_challenge_and_stack.png` — The real $3.12\,\mu\text{m}\times 3.12\,\mu\text{m}$ `ceviche_waveguide_bend` geometry (`data/round2_bend/fields_compact.npz`) and the Round-2 `FeasibleDensity` L-BFGS-B + Adam STE optimizer stack (`data/round2_bend/summary.json`).
2. `02_optimization_trace_beta_stages.png` — The 145-step Round-2 `eval_metric` and `loss` traces from `data/round2_bend/log.jsonl` across $\beta = 4 \to 8 \to 16 \to 32 \to 64\text{ (STE)}$, stage-end repaired scores, and `imageruler` length-scale trajectory from `data/round2_bend/snapshots_thinned.npz`.
3. `03_final_design_and_length_scale.png` — Our Round-2 final binary design (`data/round2_bend/design.npy`, $160\times 160$ at $10\text{ nm}$), Stage-1 $\beta=32$ density, interior violation curve vs brush diameter ($0$ violations for $d \le 8\text{ px}$), and a clearly labelled `"Toy Problem (not the challenge)"` brush-repair inset.
4. `04_fields_six_wavelengths.png` — Real Round-2 Wakis steady-state wavefields $\operatorname{Re}\{E_z(x,y;\lambda_k)\}$ and $|E_z(x,y;\lambda_k)|^2$ across all 6 challenge wavelengths from `data/round2_bend/fields_compact.npz`.
5. `05_spectrum_vs_target_windows.png` — Stored 67-wavelength Wakis dense spectrum (`fields_compact.npz`), live Wakis re-simulation (`wakis_resimulation.npz`), official `ceviche` FDFD 6-wavelength channels (`summary.json`), and `invrs-gym` leaderboard spectrum (`leaderboard_rescore.json`) vs target windows.
6. `06_wakis_resim_and_fdfd_parity.png` — Parity plot of live Wakis re-simulation vs stored `dense_S`, Wakis $r=1$ vs official `ceviche` FDFD ($\max|\Delta|S|^2| = 3.77\times 10^{-4}$ at $10\text{ nm}$, $1.03\times 10^{-4}$ at $40\text{ nm}$), and the $313\times 313$ memory ladder.
7. `07_leaderboard_comparison.png` — Like-for-like comparison of Wakis Round-2 (`+0.6427` and `+0.5793` at `8/8 px`), Round-1 pre-STE (`+0.3150` and `-0.0355` at 10 nm), and the `invrs-gym` leaderboard best (`-1.2504` at `9/7 px`).
8. `08_design_evolution.gif` — Animated GIF (< 5 MB) of the real Round-2 $160\times 160$ bend topology evolving inside the $3.12\,\mu\text{m}\times 3.12\,\mu\text{m}$ waveguide domain (`snapshots_thinned.npz` + `log.jsonl`).

In [ ]:
def _plot_00_setup_schematics(fig_dir: Path) -> None:
    """Generate 00a/00b setup figures and 00_setup_table.md (base, _initial, _final) from stored data."""
    from matplotlib.patches import Rectangle
    from wakis import figstyle, schematic
    from wakis.templates.shapes import Box

    art = load_round2_artifacts()
    design_final = np.asarray(art["design"], dtype=np.float64)
    design_init = np.asarray(art["snapshots"]["designs"][0], dtype=np.float64)
    eps_full = art["eps_full"]
    dense_wl_nm = art["dense_wavelengths_nm"]
    wl_6_nm = art["wavelengths_nm"]
    p0, p1 = art["ports"][0], art["ports"][1]

    dl = 10e-9
    nx, ny = 312, 312
    n_pml = 20
    dt = 0.95 * dl / (C0 * np.sqrt(2.0))
    f_target = C0 / (dense_wl_nm * 1e-9)
    f_sim = np.arcsin(np.pi * f_target * dt) / (np.pi * dt)
    f_6 = C0 / (wl_6_nm * 1e-9)
    fc_pulse = 0.5 * (float(f_sim.min()) + float(f_sim.max()))
    pulse_obj = GaussianSource(fc_pulse, 0.12 * fc_pulse)

    x_min_m = dl / 2
    x_max_m = nx * dl + dl / 2
    d0_m, d1_m = 0.92e-6, 2.52e-6
    w_wg_m = 0.40e-6
    yc_in_m = 1.72e-6
    xc_out_m = 1.72e-6
    offset_um = 0.05

    sources_s = [
        {
            "kind": "sheet",
            "type_name": "PolarizedModeSource",
            "short_word": "ModeSrc",
            "label": "Port 1 PolarizedModeSource (TM₀ E_z, +x)",
            "desc": f"Port 1 PolarizedModeSource (TM₀ E_z, +x) at x = {p0[0]:.2f} µm, y ∈ [{p0[1] - 0.5 * p0[2]:.2f}, {p0[1] + 0.5 * p0[2]:.2f}] µm",
            "pos_str": f"x = {p0[0]:.2f} µm, y ∈ [{p0[1] - 0.5 * p0[2]:.2f}, {p0[1] + 0.5 * p0[2]:.2f}] µm",
            "polarization": "TM₀ (E_z)",
            "h_bounds": (p0[0] * 1e-6, p0[0] * 1e-6),
            "v_bounds": ((p0[1] - 0.5 * p0[2]) * 1e-6, (p0[1] + 0.5 * p0[2]) * 1e-6),
            "e_vec": (0.0, 0.0, 1.0),
            "k_vec": (1.0, 0.0, 0.0),
            "h_vec": (0.0, -1.0, 0.0),
            "pulse": schematic._extract_pulse_metadata(pulse_obj),
        }
    ]
    monitors_s = [
        {
            "kind": "plane",
            "type_name": "PolarizedModeMonitor",
            "short_word": "S₁₁ mon",
            "label": "Port 1 PolarizedModeMonitor (S₁₁ reflection, TM₀ E_z)",
            "desc": f"Port 1 PolarizedModeMonitor (S₁₁) at x = {p0[0] + offset_um:.2f} µm, y ∈ [{p0[1] - 0.5 * p0[2]:.2f}, {p0[1] + 0.5 * p0[2]:.2f}] µm",
            "pos_str": f"x = {p0[0] + offset_um:.2f} µm, y ∈ [{p0[1] - 0.5 * p0[2]:.2f}, {p0[1] + 0.5 * p0[2]:.2f}] µm",
            "frequencies": f_6,
            "freq_str": "6 target channels λ ∈ {1265..1295} nm (67 dense λ ∈ [1245, 1315] nm)",
            "h_bounds": ((p0[0] + offset_um) * 1e-6, (p0[0] + offset_um) * 1e-6),
            "v_bounds": ((p0[1] - 0.5 * p0[2]) * 1e-6, (p0[1] + 0.5 * p0[2]) * 1e-6),
        },
        {
            "kind": "plane",
            "type_name": "PolarizedModeMonitor",
            "short_word": "S₂₁ obj",
            "label": "Port 2 Objective PolarizedModeMonitor (S₂₁ transmission, TM₀ E_z)",
            "desc": f"Port 2 Objective PolarizedModeMonitor (S₂₁) at y = {p1[1] + offset_um:.2f} µm, x ∈ [{p1[0] - 0.5 * p1[2]:.2f}, {p1[0] + 0.5 * p1[2]:.2f}] µm",
            "pos_str": f"x ∈ [{p1[0] - 0.5 * p1[2]:.2f}, {p1[0] + 0.5 * p1[2]:.2f}] µm, y = {p1[1] + offset_um:.2f} µm",
            "frequencies": f_6,
            "freq_str": "6 target channels λ ∈ {1265..1295} nm (67 dense λ ∈ [1245, 1315] nm)",
            "h_bounds": ((p1[0] - 0.5 * p1[2]) * 1e-6, (p1[0] + 0.5 * p1[2]) * 1e-6),
            "v_bounds": ((p1[1] + offset_um) * 1e-6, (p1[1] + offset_um) * 1e-6),
        },
    ]
    variations = [
        {
            "name": "Initial design (_initial)",
            "details": "Stage-1 iter 0 initial greyscale density (mean ρ ≈ 0.49, β = 4) in 160 × 160 design box Ω_d (1.60 µm × 1.60 µm)",
        },
        {
            "name": "Final Round-2 design (_final)",
            "details": "Stage-2 STE binary design (ste_beta64:34, 8/8 px = 80 nm, official eval_metric = +0.6427) from data/round2_bend/design.npy",
        },
        {
            "name": "Quick re-simulation (task_1)",
            "details": "2× coarsened 20 nm grid (156 × 156 × 1, 18.1 cells/λ_min) vs stored 10 nm reference",
        },
    ]

    variants = [
        (
            "_initial",
            design_init,
            7.25,
            "Design region Ω_d (initial ρ ≈ 0.49, 1.60 µm × 1.60 µm)",
            "Design region Ω_d\n(initial ρ ≈ 0.49)",
            "ceviche_waveguide_bend Setup — Initial Design (Stage 1, iter 0)",
            "Discretization (10 nm Challenge Grid & 1× Zoom) and O-Band Excitation — Initial",
        ),
        (
            "_final",
            design_final,
            12.25,
            "Design region Ω_d (final binary 8/8 px = 80 nm, ε ∈ {2.25, 12.25})",
            "Design region Ω_d\n(final 8/8 px = 80 nm)",
            "ceviche_waveguide_bend Setup — Final Round-2 Design (+0.6427, 8/8 px)",
            "Discretization (10 nm Challenge Grid & 1× Zoom) and O-Band Excitation — Final",
        ),
        (
            "",
            design_final,
            12.25,
            "Design region Ω_d (160×160 px, 1.60 µm × 1.60 µm, ε ∈ {2.25, 12.25})",
            "Design region Ω_d\n(hatched, 1.60 µm × 1.60 µm)",
            "ceviche_waveguide_bend — 2D Adjoint Inverse-Design Setup (10 nm Grid)",
            "Discretization (312×312 10 nm Mesh & 1× Zoom) and O-Band Excitation",
        ),
    ]

    zoom_bounds = (0.86e-6, 1.18e-6, 1.48e-6, 1.96e-6)

    for suffix, des_2d, des_eps, des_struct_name, des_box_lbl, title_a, title_b in variants:
        grid = wakis.GridFIT3D(
            x_min_m, x_max_m, x_min_m, x_max_m, -dl / 2, dl / 2, nx, ny, 1, verbose=0
        )
        solver = wakis.SolverFIT3D(
            grid,
            bc_low=["cpml", "cpml", "periodic"],
            bc_high=["cpml", "cpml", "periodic"],
            bg=[2.25, 1.0, 0.0],
            n_pml=n_pml,
            dt=dt,
            verbose=0,
        )
        eps_nodes = np.where(eps_full > 7.0, 12.25, 2.25)
        eps_nodes[92:252, 92:252] = 2.25 + (12.25 - 2.25) * des_2d
        for comp in "xyz":
            solver.ieps[:, :, 0, comp] = 1.0 / (eps_nodes * EPS_0)

        structs = [
            Box(
                center=(0.5 * (x_min_m + d0_m), yc_in_m, 0.0),
                size=(d0_m - x_min_m, w_wg_m, dl),
                material=12.25,
            ),
            Box(
                center=(xc_out_m, 0.5 * (x_min_m + d0_m), 0.0),
                size=(w_wg_m, d0_m - x_min_m, dl),
                material=12.25,
            ),
            Box(
                center=(0.5 * (d0_m + d1_m), 0.5 * (d0_m + d1_m), 0.0),
                size=(d1_m - d0_m, d1_m - d0_m, dl),
                material=des_eps,
            ),
        ]
        mat_names = {
            "background": "SiO₂ cladding (ε = 2.25, n = 1.50)",
            "region_0": "Input Si waveguide (ε = 12.25, n = 3.50, w = 400 nm)",
            "region_1": "Output Si waveguide (ε = 12.25, n = 3.50, w = 400 nm)",
            "region_2": des_struct_name,
        }
        res = schematic.save_setup_figures(
            solver,
            fig_dir,
            prefix="00",
            suffix=suffix,
            plane="xy",
            structures=structs,
            material_names=mat_names,
            sources=sources_s,
            monitors=monitors_s,
            unit="µm",
            wavelength=1265e-9,
            min_feature=80e-9,
            title_geometry=title_a,
            title_mesh=title_b,
            variations=variations,
        )

        # Enhance 00a: render design array inside hatched design box and place tags cleanly
        fig_a, ax_a = schematic.plot_schematic(
            solver,
            plane="xy",
            structures=structs[:2],
            material_names=mat_names,
            sources=sources_s,
            monitors=monitors_s,
            unit="µm",
            wavelength=1265e-9,
            title=title_a,
            width_in=7.6,
            height_in=4.9,
        )
        eps_des = 2.25 + (12.25 - 2.25) * des_2d.T
        ax_a.imshow(
            eps_des,
            origin="lower",
            extent=[0.92, 2.52, 0.92, 2.52],
            cmap="Blues",
            vmin=2.25,
            vmax=12.25,
            alpha=0.84,
            zorder=3.2,
            interpolation="nearest",
        )
        ax_a.add_patch(
            Rectangle(
                (0.92, 0.92),
                1.60,
                1.60,
                facecolor="none",
                edgecolor="#6D28D9",
                hatch="xx",
                lw=1.6,
                ls="--",
                zorder=5.4,
            )
        )
        ax_a.text(
            1.72,
            2.56,
            des_box_lbl,
            ha="center",
            va="bottom",
            fontsize=7.8,
            fontweight="bold",
            color="#5B21B6",
            bbox=dict(boxstyle="round,pad=0.16", facecolor="#F5F3FF", edgecolor="#6D28D9", lw=0.8, alpha=0.94),
            zorder=9,
        )
        for txt_obj in ax_a.texts:
            t_str = txt_obj.get_text()
            role = getattr(txt_obj, "_schematic_role", "")
            if t_str.startswith("1 ·"):
                txt_obj.set_position((0.34, 2.18))
                txt_obj.set_fontsize(8.0)
                txt_obj.set_ha("left")
                txt_obj.set_va("center")
                txt_obj.set_bbox(dict(boxstyle="round,pad=0.12", facecolor="white", edgecolor="none", alpha=0.88))
                txt_obj.set_zorder(9)
            elif t_str.startswith("2 ·"):
                txt_obj.set_position((0.34, 0.94))
                txt_obj.set_fontsize(8.2)
                txt_obj.set_ha("left")
                txt_obj.set_va("top")
            elif t_str.startswith("3 ·"):
                txt_obj.set_position((2.44, 0.40))
                txt_obj.set_fontsize(8.2)
                txt_obj.set_ha("right")
                txt_obj.set_va("bottom")
            elif role == "boundary_bottom":
                txt_obj.set_position((0.85, -0.08))
                txt_obj.set_ha("center")
                txt_obj.set_va("top")
        ax_a.spines["bottom"].set_position(("outward", 24))

        leg = ax_a.get_legend()
        if leg is not None:
            handles = list(leg.legend_handles)
            labels = [t.get_text() for t in leg.get_texts()]
            handles.insert(
                2,
                Rectangle(
                    (0, 0),
                    1,
                    1,
                    facecolor="#DBEAFE",
                    edgecolor="#6D28D9",
                    hatch="xx",
                    lw=1.2,
                    ls="--",
                ),
            )
            labels.insert(2, f"{des_struct_name} (d = 1.60 µm)")
            ax_a.legend(
                handles,
                labels,
                loc="upper center",
                bbox_to_anchor=(0.5, -0.28),
                ncol=2,
                frameon=False,
                fontsize=8.6,
                handlelength=1.8,
                columnspacing=1.2,
            )
        figstyle.save(fig_a, res["geometry_figure"], close=True, auto_legend_below=False)

        # Enhance 00b: overlay design array and hatched design region box on mesh panel & zoom inset
        fig_b, axes_b = schematic.plot_mesh_and_excitation(
            solver,
            plane="xy",
            structures=structs[:2],
            material_names=mat_names,
            sources=sources_s,
            monitors=monitors_s,
            unit="µm",
            mesh_every=8,
            zoom_bounds=zoom_bounds,
            wavelength=1265e-9,
            min_feature=80e-9,
            title=title_b,
            width_in=7.6,
            height_in=5.6,
        )
        ax_mesh = axes_b[0]
        ax_mesh.imshow(
            eps_des,
            origin="lower",
            extent=[0.92, 2.52, 0.92, 2.52],
            cmap="Blues",
            vmin=2.25,
            vmax=12.25,
            alpha=0.74,
            zorder=2.1,
            interpolation="nearest",
        )
        ax_mesh.add_patch(
            Rectangle(
                (0.92, 0.92),
                1.60,
                1.60,
                facecolor="none",
                edgecolor="#6D28D9",
                hatch="xx",
                lw=1.3,
                ls="--",
                zorder=3.2,
            )
        )
        for child_ax in ax_mesh.child_axes:
            child_ax.set_axes_locator(None)
            child_ax.set_position([0.68, 0.56, 0.24, 0.28])
            child_ax.imshow(
                eps_des,
                origin="lower",
                extent=[0.92, 2.52, 0.92, 2.52],
                cmap="Blues",
                vmin=2.25,
                vmax=12.25,
                alpha=0.74,
                zorder=1.6,
                interpolation="nearest",
            )
            child_ax.add_patch(
                Rectangle(
                    (0.92, 0.92),
                    1.60,
                    1.60,
                    facecolor="none",
                    edgecolor="#6D28D9",
                    hatch="xx",
                    lw=1.1,
                    ls="--",
                    zorder=2.2,
                )
            )
            for ins_txt in child_ax.texts:
                ins_txt.set_fontsize(8.4)
                ins_txt.set_position((0.5, 0.93))
        figstyle.save(fig_b, res["mesh_figure"], close=True, auto_legend_below=False)


def _draw_round2_domain(ax, art: dict, design_2d: np.ndarray, title: str = "") -> None:
    from matplotlib.colors import LinearSegmentedColormap
    import matplotlib.patches as mpatches

    cmap_mat = LinearSegmentedColormap.from_list("material", ["#eceef4", "#4f63d2"])
    eps_bg = np.where(art["eps_full"] > 7.0, 1.0, 0.0)
    eps_bg[92:252, 92:252] = np.asarray(design_2d, dtype=np.float64)
    x_um, y_um = art["x_um"], art["y_um"]
    ext = [float(x_um.min()), float(x_um.max()), float(y_um.min()), float(y_um.max())]
    ax.imshow(eps_bg.T, origin="lower", extent=ext, cmap=cmap_mat, vmin=0.0, vmax=1.0, interpolation="nearest")

    pml = art["pml_um"]
    kw = dict(facecolor="black", alpha=0.08, lw=0)
    ax.add_patch(mpatches.Rectangle((ext[0], ext[2]), pml, ext[3] - ext[2], **kw))
    ax.add_patch(mpatches.Rectangle((ext[1] - pml, ext[2]), pml, ext[3] - ext[2], **kw))
    ax.add_patch(mpatches.Rectangle((ext[0] + pml, ext[2]), ext[1] - ext[0] - 2 * pml, pml, **kw))
    ax.add_patch(mpatches.Rectangle((ext[0] + pml, ext[3] - pml), ext[1] - ext[0] - 2 * pml, pml, **kw))

    d_um = art["design_um"]
    ax.add_patch(
        mpatches.Rectangle(
            (d_um[0], d_um[1]),
            d_um[2] - d_um[0],
            d_um[3] - d_um[1],
            fill=False,
            edgecolor="#d62728",
            lw=1.2,
            ls="--",
        )
    )
    for k, (px, py, pw, axis, sign) in enumerate(art["ports"]):
        if int(axis) == 0:
            ax.plot([px, px], [py - pw / 2, py + pw / 2], color="#e0549b", lw=1.8)
            if k == 0:
                ax.annotate(
                    "",
                    xy=(px + 0.38 * sign, py),
                    xytext=(px, py),
                    arrowprops=dict(arrowstyle="-|>", color="#e0549b", lw=1.5),
                )
        else:
            ax.plot([px - pw / 2, px + pw / 2], [py, py], color="#e0549b", lw=1.8)
    ax.set_aspect("equal")
    ax.set_xlabel(r"$x$ [$\mu$m]")
    ax.set_ylabel(r"$y$ [$\mu$m]")
    if title:
        ax.set_title(title, fontsize=9.5)


def _plot_01_challenge_and_stack(fig_dir: Path) -> None:
    import matplotlib.patches as mpatches
    import matplotlib.pyplot as plt

    art = load_round2_artifacts()
    fig = plt.figure(figsize=(12.2, 4.8), constrained_layout=True)
    gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.55])

    ax_g = fig.add_subplot(gs[0, 0])
    _draw_round2_domain(
        ax_g,
        art,
        art["design"],
        title="Round-2 `ceviche_waveguide_bend` Geometry\n($312\\times 312$ at 10 nm, $160\\times 160$ design box)",
    )
    ax_g.text(0.45, 2.05, "Port 1 (in)\n1265..1295 nm", fontsize=8.0, color="#b71c1c", fontweight="bold")
    ax_g.text(1.75, 0.42, "Port 2 (out)", fontsize=8.0, color="#b71c1c", fontweight="bold")
    ax_g.text(
        1.72,
        1.72,
        "Round-2 Design\n$1.6\\times 1.6\\,\\mu$m\n(8/8 px = 80 nm)",
        ha="center",
        va="center",
        fontsize=8.0,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.82, edgecolor="#d62728"),
    )

    ax = fig.add_subplot(gs[0, 1])
    ax.set_xlim(0, 7.2)
    ax.set_ylim(0, 5.0)
    ax.axis("off")
    ax.text(
        3.6,
        4.68,
        "Round-2 `invrs-wakis` Two-Stage FeasibleDensity + STE Optimizer Stack",
        ha="center",
        va="center",
        fontsize=11.0,
        fontweight="bold",
    )

    s1_box = mpatches.FancyBboxPatch(
        (0.2, 2.55),
        6.8,
        1.8,
        boxstyle="round,pad=0.1",
        facecolor="#edf7ed",
        edgecolor="#2e7d32",
        lw=1.6,
    )
    ax.add_patch(s1_box)
    ax.text(
        3.6,
        4.02,
        r"Stage 1 · Density L-BFGS-B across $\beta = 4 \to 8 \to 16 \to 32$ (30 + 30 + 25 + 20 = 105 steps)",
        ha="center",
        va="center",
        fontsize=9.5,
        fontweight="bold",
        color="#1b5e20",
    )
    ax.text(
        3.6,
        3.22,
        r"1. Diagonal symmetry + conic filter ($R = 8$ px) + Wang–Lazarov–Sigmund $\tanh(\beta(\tilde{\rho}-0.5))$ projection"
        + "\n"
        + r"2. Broadband Wakis FIT forward + exact discrete adjoint VJP (`match_fdfd=True`, $r=1$, 6 wavelengths)"
        + "\n"
        + r"3. Stage-end `imageruler` repair $\to$ `CandidatePool` (Stage 1 $\beta=32$: continuous $+0.5585 \to$ repaired $+0.1987$)",
        ha="center",
        va="center",
        fontsize=8.3,
    )

    s2_box = mpatches.FancyBboxPatch(
        (0.2, 0.4),
        6.8,
        1.8,
        boxstyle="round,pad=0.1",
        facecolor="#fce8e6",
        edgecolor="#c62828",
        lw=1.6,
    )
    ax.add_patch(s2_box)
    ax.text(
        3.6,
        1.88,
        r"Stage 2 · Morphological Brush Repair + Straight-Through Estimator ($\beta = 64$, 40 Adam steps)",
        ha="center",
        va="center",
        fontsize=9.5,
        fontweight="bold",
        color="#b71c1c",
    )
    ax.text(
        3.6,
        1.05,
        r"1. Forward pass simulates repaired binary design (`imageruler` opening/closing guarantees $8/8$ px $= 80$ nm)"
        + "\n"
        + r"2. Backward pass pulls exact Wakis adjoint sensitivity through smooth $\beta=64$ projection ($\mathrm{lr}=0.02$)"
        + "\n"
        + r"3. Selected `ste_beta64:34` (iter 139): Wakis $+0.6412$, official `ceviche` FDFD $+0.6427$ at $8/8$ px",
        ha="center",
        va="center",
        fontsize=8.3,
    )
    ax.annotate("", xy=(3.6, 2.25), xytext=(3.6, 2.52), arrowprops=dict(arrowstyle="->", lw=1.8, color="#333333"))

    fig.savefig(fig_dir / "01_bend_challenge_and_stack.png", dpi=180)
    plt.close(fig)


def _plot_02_optimization_trace(fig_dir: Path) -> None:
    import matplotlib.pyplot as plt

    ls_data = np.load(DATA_DIR / "length_scale_check.npz")
    log_loss = ls_data["log_loss"]
    log_eval = ls_data["log_eval"]
    log_stage = ls_data["log_stage"]
    repaired_evals = ls_data["repaired_evals"]
    snap_idx = ls_data["snap_kept_index"]
    snap_bin = ls_data["snap_binarization"]
    snap_mw = ls_data["snap_min_width"]
    snap_ms = ls_data["snap_min_space"]

    iters = np.arange(len(log_loss))
    fig, axes = plt.subplots(1, 3, figsize=(13.8, 4.3), constrained_layout=True)

    stage_Order = ["density_beta4", "density_beta8", "density_beta16", "density_beta32", "ste_beta64"]
    stage_labels = [r"$\beta=4$", r"$\beta=8$", r"$\beta=16$", r"$\beta=32$", r"STE $\beta=64$"]
    stage_cols = ["#e8f4f8", "#e0f2f1", "#fff3e0", "#fbe9e7", "#f3e5f5"]
    stage_ends = []

    ax0 = axes[0]
    for st_name, lbl, col in zip(stage_Order, stage_labels, stage_cols):
        mask = np.where(log_stage == st_name)[0]
        if len(mask):
            i0, i1 = int(mask[0]), int(mask[-1])
            stage_ends.append(i1)
            ax0.axvspan(i0 - 0.5, i1 + 0.5, color=col, alpha=0.65, label=lbl)

    ax0.plot(iters, log_eval, "-", color="#1f77b4", lw=1.8, label="Wakis `eval_metric`")
    ax0.scatter(
        stage_ends,
        repaired_evals,
        color="#d62728",
        marker="D",
        s=42,
        zorder=5,
        label="Stage-end repaired (8/8 px)",
    )
    # Selected candidate ste_beta64 step 34 = iteration 105 + 34 = 139
    ax0.scatter(
        [139],
        [log_eval[139]],
        color="#2ca02c",
        marker="*",
        s=150,
        edgecolor="k",
        zorder=6,
        label=f"Selected iter 139 (+{log_eval[139]:.4f})",
    )
    ax0.axhline(0.0, color="#444444", ls=":", lw=1.0)
    ax0.set_ylim(-2.2, 0.85)
    ax0.set_xlabel("Optimization iteration (Round-2 `log.jsonl`)")
    ax0.set_ylabel("Wakis `eval_metric`")
    ax0.set_title(r"(a) Round-2 `eval_metric` Across $\beta=4..32$ & STE $\beta=64$")
    ax0.grid(True, alpha=0.3)
    ax0.legend(fontsize=7.3, ncol=2, loc="lower right")

    ax1 = axes[1]
    ax1.semilogy(iters, log_loss, "-", color="#d62728", lw=1.8, label="Round-2 objective `loss`")
    ax1.axvline(104.5, color="#333333", ls="--", lw=1.2)
    ax1_r = ax1.twinx()
    ax1_r.plot(snap_idx, snap_bin * 100.0, "s--", color="#333333", lw=1.3, ms=3.5, label="Binarization [%]")
    ax1_r.set_ylabel("Binarization degree [%]", color="#333333")
    ax1_r.set_ylim(0, 106)
    ax1.set_xlabel("Optimization iteration")
    ax1.set_ylabel("Objective loss (log scale)", color="#d62728")
    ax1.set_title("(b) Round-2 Loss & 100% Binarization in STE Stage")
    ax1.grid(True, alpha=0.3)

    ax2 = axes[2]
    ax2.plot(snap_idx, snap_mw, "o-", color="#1f77b4", lw=1.6, ms=3.8, label="Min solid width [px]")
    ax2.plot(snap_idx, snap_ms, "s--", color="#ff7f0e", lw=1.6, ms=3.8, label="Min void spacing [px]")
    ax2.axhline(8.0, color="#2ca02c", ls=":", lw=1.6, label="Target 8 px (80 nm)")
    ax2.axvline(104.5, color="#333333", ls="--", lw=1.2, label="Enter STE (iter 105)")
    ax2.set_ylim(0, 10.5)
    ax2.set_xlabel("Optimization iteration (`snapshots_thinned.npz`)")
    ax2.set_ylabel("Thresholded feature size [px at 10 nm]")
    ax2.set_title("(c) Length-Scale Guarantee in Stage-2 STE (8/8 px)")
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=7.8, loc="lower right")

    fig.savefig(fig_dir / "02_optimization_trace_beta_stages.png", dpi=180)
    plt.close(fig)


def _plot_03_final_design_and_length_scale(fig_dir: Path) -> None:
    import matplotlib.patches as mpatches
    import matplotlib.pyplot as plt

    art = load_round2_artifacts()
    ls_data = np.load(DATA_DIR / "length_scale_check.npz")
    toy_data = np.load(DATA_DIR / "toy_bend_demo.npz")
    snaps = art["snapshots"]["designs"].astype(np.float32)
    rho_stage1 = snaps[35]  # iter 102 (density_beta32)
    des_fin = art["design"]

    fig, axes = plt.subplots(1, 4, figsize=(14.6, 3.9), constrained_layout=True)

    # (a) Stage-1 beta=32 density before STE repair
    im0 = axes[0].imshow(rho_stage1.T, origin="lower", cmap="viridis", vmin=0.0, vmax=1.0)
    axes[0].set_title(
        r"(a) Round-2 Stage-1 Density ($\beta=32$)" + "\nThresholded: 3/5 px (violates 8 px)",
        fontsize=9.4,
    )
    axes[0].set_xlabel("Design pixel $i_x$ (10 nm)")
    axes[0].set_ylabel("Design pixel $i_y$ (10 nm)")
    fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    # (b) Final Round-2 binary design (`round2_bend/design.npy`, 160x160)
    im1 = axes[1].imshow(des_fin.T, origin="lower", cmap="Greys", vmin=0.0, vmax=1.0)
    circ = mpatches.Circle(
        (22.0, 138.0),
        radius=4.0,
        facecolor="none",
        edgecolor="#d62728",
        lw=2.0,
        label="Brush $d=8$ px (80 nm)",
    )
    axes[1].add_patch(circ)
    axes[1].set_title(
        "(b) Round-2 Final Design (`design.npy`)\nOfficial $+0.6427$, Measured: 8/8 px",
        fontsize=9.4,
    )
    axes[1].set_xlabel("Design pixel $i_x$ (10 nm)")
    axes[1].set_ylabel("Design pixel $i_y$ (10 nm)")
    axes[1].legend(loc="upper right", fontsize=7.8)
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    # (c) Interior length-scale violation count vs brush diameter d on round2_bend/design.npy
    diams = ls_data["diameters"]
    v_sol = ls_data["solid_violations"]
    v_void = ls_data["void_violations"]
    axes[2].plot(diams, v_sol, "o-", color="#1f77b4", lw=1.8, label="Solid interior violations")
    axes[2].plot(diams, v_void, "s--", color="#ff7f0e", lw=1.8, label="Void interior violations")
    axes[2].axvline(8.0, color="#2ca02c", ls=":", lw=1.6, label="Feasible limit $d=8$ px (0 violations)")
    axes[2].set_xlabel("Brush diameter $d$ [px at 10 nm]")
    axes[2].set_ylabel("Interior violation pixels")
    axes[2].set_title("(c) Length-Scale Check on `round2_bend/design.npy`\n(0 violations for $d \\leq 8$ px)", fontsize=9.4)
    axes[2].grid(True, alpha=0.3)
    axes[2].legend(fontsize=7.8, loc="upper left")

    # (d) Toy problem (not the challenge) inset
    im3 = axes[3].imshow(toy_data["toy_binary"].T, origin="lower", cmap="Greys", vmin=0.0, vmax=1.0)
    axes[3].set_title(
        f"(d) Toy Problem (not the challenge)\n16x16 live CPU demo (`toy_bend_demo.npz`)",
        fontsize=9.4,
    )
    axes[3].set_xlabel("Toy pixel $i_x$")
    axes[3].set_ylabel("Toy pixel $i_y$")
    fig.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

    fig.savefig(fig_dir / "03_final_design_and_length_scale.png", dpi=180)
    plt.close(fig)


def _plot_04_fields_six_wavelengths(fig_dir: Path) -> None:
    import matplotlib.patches as mpatches
    import matplotlib.pyplot as plt

    art = load_round2_artifacts()
    x_um, y_um = art["x_um"], art["y_um"]
    pml = art["pml_um"]
    sx = (x_um > x_um.min() + pml) & (x_um < x_um.max() - pml)
    sy = (y_um > y_um.min() + pml) & (y_um < y_um.max() - pml)
    xc, yc = x_um[sx], y_um[sy]
    eps_c = art["eps_full"][np.ix_(sx, sy)]
    ez_c = art["ez_full"][:, sx][:, :, sy]
    wls = art["wavelengths_nm"]
    s2_off = np.array(art["summary"]["official_S2"], dtype=np.float64)[:, 0, :]
    s21_db = 10.0 * np.log10(s2_off[:, 1])
    d_um = art["design_um"]
    ext = [float(xc.min()), float(xc.max()), float(yc.min()), float(yc.max())]

    fig, axes = plt.subplots(2, 3, figsize=(13.4, 8.0), constrained_layout=True)
    vmax = float(np.percentile(np.abs(ez_c), 99.6))

    for k in range(6):
        ax = axes[k // 3, k % 3]
        im = ax.imshow(
            ez_c[k].real.T,
            origin="lower",
            extent=ext,
            cmap="RdBu_r",
            vmin=-vmax,
            vmax=vmax,
        )
        ax.contour(xc, yc, eps_c.T, levels=[7.25], colors="k", linewidths=0.65)
        ax.add_patch(
            mpatches.Rectangle(
                (d_um[0], d_um[1]),
                d_um[2] - d_um[0],
                d_um[3] - d_um[1],
                fill=False,
                edgecolor="#222222",
                lw=1.0,
                ls="--",
            )
        )
        ax.set_title(
            rf"$\lambda_{k + 1} = {wls[k]:.0f}\,\mathrm{{nm}}$: $\mathrm{{Re}}\,E_z$ ($|S_{{21}}|^2 = {s21_db[k]:.3f}\,\mathrm{{dB}}$)",
            fontsize=9.5,
        )
        ax.set_xlabel(r"$x$ [$\mu$m]")
        ax.set_ylabel(r"$y$ [$\mu$m]")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig.suptitle(
        "Round-2 `ceviche_waveguide_bend` Steady-State Wakis DFT Fields (`data/round2_bend/fields_compact.npz`)",
        fontsize=11.0,
        fontweight="bold",
    )
    fig.savefig(fig_dir / "04_fields_six_wavelengths.png", dpi=180)
    plt.close(fig)


def _plot_05_spectrum_vs_windows(fig_dir: Path) -> None:
    import matplotlib.pyplot as plt

    resim = np.load(DATA_DIR / "wakis_resimulation.npz")
    with open(DATA_DIR / "round2_evaluation_summary.json") as f:
        eval_summ = json.load(f)

    wls_dense = resim["dense_wavelengths_nm"]
    wls_6 = resim["wavelengths_6_nm"]
    s21_stored_db = 10.0 * np.log10(np.maximum(resim["s21_sq_stored"], 1e-6))
    s21_resim_db = 10.0 * np.log10(np.maximum(resim["s21_sq_resim"], 1e-6))
    s11_stored_db = 10.0 * np.log10(np.maximum(resim["s11_sq_stored"], 1e-6))
    s11_resim_db = 10.0 * np.log10(np.maximum(resim["s11_sq_resim"], 1e-6))

    s2_off = resim["s2_official_6"]
    s2_seed_a = np.array(eval_summ["leaderboard_rescore"]["wakis_round2_seed_a"]["s2"])
    s2_lb = np.array(eval_summ["leaderboard_rescore"]["invrs_gym_leaderboard_best"]["s2"])

    fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.6), constrained_layout=True)

    ax0 = axes[0]
    ax0.axvspan(wls_6[0], wls_6[-1], color="#e8f5e9", alpha=0.55, label="Target band (1265..1295 nm)")
    ax0.axhline(-0.5, color="#2e7d32", ls="--", lw=1.4, label="Target floor ($-0.5\\,\\mathrm{dB}$)")
    ax0.plot(wls_dense, s21_stored_db, "-", color="#1f77b4", lw=2.2, label="Stored Wakis `dense_S` (`fields_compact.npz`)")
    ax0.plot(wls_dense, s21_resim_db, "--", color="#ff7f0e", lw=1.7, label="Live Wakis re-simulation (`wakis_resimulation.npz`)")
    ax0.scatter(wls_6, 10.0 * np.log10(s2_off[:, 1]), color="#2ca02c", marker="o", s=55, zorder=5, label="Official `ceviche` FDFD (+0.6427)")
    ax0.scatter(wls_6, 10.0 * np.log10(s2_seed_a[:, 1]), color="#9467bd", marker="s", s=45, zorder=5, label="Wakis Round-2 Seed A (+0.5793)")
    ax0.plot(wls_6, 10.0 * np.log10(s2_lb[:, 1]), "v-.", color="#d62728", lw=1.4, ms=5.5, label="Leaderboard best (-1.2504, 9/7 px)")
    ax0.set_ylim(-0.62, 0.02)
    ax0.set_xlabel(r"Wavelength $\lambda$ [nm]")
    ax0.set_ylabel(r"Transmission $10\log_{10}|S_{21}(\lambda)|^2$ [dB]")
    ax0.set_title(r"(a) Broadband Transmission $|S_{21}(\lambda)|^2$ vs. $[-0.5, 0.0]\,\mathrm{dB}$ Window")
    ax0.grid(True, alpha=0.3)
    ax0.legend(fontsize=7.6, loc="lower left")

    ax1 = axes[1]
    ax1.axvspan(wls_6[0], wls_6[-1], color="#ffebee", alpha=0.45, label="Target band (1265..1295 nm)")
    ax1.axhline(-20.0, color="#c62828", ls="--", lw=1.4, label="Target reflection ceiling ($-20\\,\\mathrm{dB}$)")
    ax1.axhline(-31.0, color="#555555", ls=":", lw=1.2, label="`ceviche` port reflection floor ($-31\\,\\mathrm{dB}$)")
    ax1.plot(wls_dense, s11_stored_db, "-", color="#1f77b4", lw=2.0, label="Stored Wakis `dense_S` (`fields_compact.npz`)")
    ax1.plot(wls_dense, s11_resim_db, "--", color="#ff7f0e", lw=1.7, label="Live Wakis re-simulation (`wakis_resimulation.npz`)")
    ax1.scatter(wls_6, 10.0 * np.log10(s2_off[:, 0]), color="#2ca02c", marker="o", s=55, zorder=5, label="Official `ceviche` FDFD (+0.6427)")
    ax1.plot(wls_6, 10.0 * np.log10(s2_lb[:, 0]), "v-.", color="#d62728", lw=1.5, ms=5.5, label="Leaderboard best (-16.48 dB > -20 dB!)")
    ax1.set_ylim(-60.0, -12.0)
    ax1.set_xlabel(r"Wavelength $\lambda$ [nm]")
    ax1.set_ylabel(r"Return Loss $10\log_{10}|S_{11}(\lambda)|^2$ [dB]")
    ax1.set_title(r"(b) Broadband Reflection $|S_{11}(\lambda)|^2$ vs. $\leq -20\,\mathrm{dB}$ Window")
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=7.6, loc="upper right")

    fig.savefig(fig_dir / "05_spectrum_vs_target_windows.png", dpi=180)
    plt.close(fig)


def _plot_06_resim_and_fdfd_parity(fig_dir: Path) -> None:
    import matplotlib.pyplot as plt

    resim = np.load(DATA_DIR / "wakis_resimulation.npz")
    with open(DATA_DIR / "round2_evaluation_summary.json") as f:
        summ = json.load(f)

    fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.5), constrained_layout=True)

    ax0 = axes[0]
    wls_dense = resim["dense_wavelengths_nm"]
    wls_6 = resim["wavelengths_6_nm"]
    diff_resim_s21 = np.abs(resim["s21_sq_resim"] - resim["s21_sq_stored"])
    diff_resim_s11 = np.abs(resim["s11_sq_resim"] - resim["s11_sq_stored"])
    diff_fdfd_cev = np.abs(resim["s2_official_6"][:, 1] - resim["s2_wakis_cev_6"][:, 1])
    diff_fdfd_mod = np.abs(resim["s2_official_6"][:, 1] - resim["s2_wakis_mod_6"][:, 1])

    ax0.semilogy(
        wls_dense,
        np.maximum(diff_resim_s21, 1e-7),
        "-",
        color="#1f77b4",
        lw=1.8,
        label=f"Live Wakis re-sim vs. stored $|S_{{21}}|^2$ (max={float(resim['max_abs_diff_s21_dense']):.2e})",
    )
    ax0.semilogy(
        wls_dense,
        np.maximum(diff_resim_s11, 1e-7),
        "--",
        color="#ff7f0e",
        lw=1.6,
        label=f"Live Wakis re-sim vs. stored $|S_{{11}}|^2$ (max={float(resim['max_abs_diff_s11_dense']):.2e})",
    )
    ax0.scatter(
        wls_6,
        diff_fdfd_cev,
        color="#2ca02c",
        s=60,
        marker="o",
        zorder=5,
        label=f"Wakis $r=1$ (cev est) vs. `ceviche` FDFD (max={float(np.max(diff_fdfd_cev)):.2e})",
    )
    ax0.scatter(
        wls_6,
        diff_fdfd_mod,
        color="#d62728",
        s=55,
        marker="^",
        zorder=5,
        label=f"Wakis $r=1$ (modal) vs. `ceviche` FDFD (max={float(np.max(diff_fdfd_mod)):.2e})",
    )
    ax0.set_xlabel(r"Wavelength $\lambda$ [nm]")
    ax0.set_ylabel(r"Absolute Power Discrepancy $|\Delta |S_{ij}(\lambda)|^2|$")
    ax0.set_title("(a) Wakis Re-Simulation Parity & `ceviche` FDFD Cross-Validation")
    ax0.grid(True, which="both", alpha=0.3)
    ax0.legend(fontsize=7.6, loc="lower right")

    ax1 = axes[1]
    proj = summ.get("memory_projection_313x313_56587steps", {})
    if proj and "ladder" in proj:
        lad = proj["ladder"]
        Nt_full = float(proj["Nt"])
        curve = lad["checkpoint_curve"]
        gb_arr = [c["bytes"] / 1e9 for c in curve]
        extra_arr = [c["extra"] / Nt_full for c in curve]
        ax1.plot(gb_arr, extra_arr, "o-", color="#ff7f0e", lw=1.8, ms=4.5, label="`checkpoint` (revolve DP curve)")
        ax1.scatter(
            [lad["store_f64"]["bytes"] / 1e9],
            [0.0],
            s=110,
            marker="s",
            color="#1f77b4",
            edgecolor="k",
            zorder=5,
            label=f"`store` float64 ({lad['store_f64']['bytes']/1e9:.2f} GB)",
        )
        ax1.scatter(
            [lad["store_f32"]["bytes"] / 1e9],
            [0.0],
            s=110,
            marker="D",
            color="#9467bd",
            edgecolor="k",
            zorder=5,
            label=f"`store` float32 ({lad['store_f32']['bytes']/1e9:.2f} GB)",
        )
        ax1.scatter(
            [lad["reverse"]["bytes"] / 1e9],
            [0.0],
            s=140,
            marker="*",
            color="#2ca02c",
            edgecolor="k",
            zorder=5,
            label=f"`reverse` CPML strip ({lad['reverse']['bytes']/1e9:.2f} GB, 0 extra)",
        )
        ax1.set_xscale("log")
        ax1.set_xlabel("Peak Memory Budget [GB]")
        ax1.set_ylabel(r"Extra Forward Passes $N_{\mathrm{recompute}} / N_t$")
        ax1.set_title(r"(b) `ceviche_waveguide_bend` ($313^2$) Adjoint Memory Ladder")
        ax1.grid(True, which="both", alpha=0.3)
        ax1.legend(fontsize=8.0, loc="upper right")

    fig.savefig(fig_dir / "06_wakis_resim_and_fdfd_parity.png", dpi=180)
    plt.close(fig)


def _plot_07_leaderboard_comparison(fig_dir: Path) -> None:
    import matplotlib.pyplot as plt

    with open(DATA_DIR / "round2_evaluation_summary.json") as f:
        summ = json.load(f)
    lb = summ["leaderboard_rescore"]

    entries = [
        ("Wakis R2 Best\n(Seed 1 rerun)\n[8/8 px Feasible]", lb["wakis_round2_best"]["eval_metric"], "#2ca02c"),
        ("Wakis R2 Seed A\n(Claim C55)\n[8/8 px Feasible]", lb["wakis_round2_seed_a"]["eval_metric"], "#4caf50"),
        ("Wakis R1 Seed 1\n(10 nm re-eval)\n[2/1 px @ 40 nm]", lb["wakis_round1_seed1"]["eval_metric_10nm"], "#ff9800"),
        ("Wakis R1 Seed 0\n(10 nm re-eval)\n[Pre-STE]", lb["wakis_round1_seed0"]["eval_metric_10nm"], "#ffb74d"),
        ("invrs-gym\nLeaderboard Best\n[9/7 px Violates]", lb["invrs_gym_leaderboard_best"]["eval_metric"], "#d62728"),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.5), constrained_layout=True)

    ax0 = axes[0]
    labels = [e[0] for e in entries]
    scores = [e[1] for e in entries]
    cols = [e[2] for e in entries]
    x = np.arange(len(entries))
    bars = ax0.bar(x, scores, color=cols, edgecolor="#222222", width=0.56)
    ax0.axhline(0.0, color="#222222", lw=1.2)
    ax0.set_xticks(x)
    ax0.set_xticklabels(labels, fontsize=8.0)
    ax0.set_ylabel("Official 10 nm `ceviche` FDFD `eval_metric`")
    ax0.set_ylim(-1.55, 0.92)
    ax0.set_title("(a) Like-for-Like `ceviche_waveguide_bend` Score Comparison")
    ax0.grid(True, axis="y", alpha=0.3)
    for b, s in zip(bars, scores):
        y_pos = s + 0.04 if s >= 0 else s - 0.14
        ax0.text(
            b.get_x() + b.get_width() / 2,
            y_pos,
            f"{s:+.4f}",
            ha="center",
            va="bottom",
            fontsize=8.6,
            fontweight="bold",
        )

    ax1 = axes[1]
    short_lbls = ["Wakis R2\n(Best)", "Wakis R2\n(Seed A)", "Leaderboard\nBest"]
    solids = [
        lb["wakis_round2_best"]["min_width_px"],
        lb["wakis_round2_seed_a"]["min_width_px"],
        lb["invrs_gym_leaderboard_best"]["min_width_px"],
    ]
    voids = [
        lb["wakis_round2_best"]["min_spacing_px"],
        lb["wakis_round2_seed_a"]["min_spacing_px"],
        lb["invrs_gym_leaderboard_best"]["min_spacing_px"],
    ]
    x1 = np.arange(len(short_lbls))
    w_bar = 0.34
    ax1.bar(x1 - w_bar / 2, solids, width=w_bar, color="#1f77b4", edgecolor="#111111", label="Min solid width [px]")
    ax1.bar(x1 + w_bar / 2, voids, width=w_bar, color="#ff7f0e", edgecolor="#111111", label="Min void spacing [px]")
    ax1.axhline(8.0, color="#2ca02c", ls="--", lw=1.6, label="Target length scale ($8\\,\\mathrm{px} = 80\\,\\mathrm{nm}$)")
    ax1.set_xticks(x1)
    ax1.set_xticklabels(short_lbls, fontsize=9.0)
    ax1.set_ylim(0, 11)
    ax1.set_ylabel("Imageruler Feature Size [px at 10 nm]")
    ax1.set_title("(b) Imageruler Solid Width & Void Spacing (`leaderboard_rescore.json`)")
    ax1.grid(True, axis="y", alpha=0.3)
    ax1.legend(fontsize=8.2, loc="upper right")

    fig.savefig(fig_dir / "07_leaderboard_comparison.png", dpi=180)
    plt.close(fig)


def _plot_08_design_evolution_gif(fig_dir: Path) -> None:
    from matplotlib.animation import FuncAnimation, PillowWriter
    from matplotlib.colors import LinearSegmentedColormap
    import matplotlib.patches as mpatches
    import matplotlib.pyplot as plt

    art = load_round2_artifacts()
    snaps = art["snapshots"]
    designs = snaps["designs"].astype(np.float64)
    kept_idx = snaps["kept_index"].astype(np.int32)
    stages = snaps["stage"]

    iter_rows = [r for r in art["log"] if "step" in r]
    log_eval = np.array([float(r["eval_metric"]) for r in iter_rows], dtype=np.float64)
    n_total_iters = len(log_eval)
    n_frames = len(designs)

    cmap_mat = LinearSegmentedColormap.from_list("material", ["#eceef4", "#4f63d2"])
    eps_bg_base = np.where(art["eps_full"] > 7.0, 1.0, 0.0)
    x_um, y_um = art["x_um"], art["y_um"]
    ext = [float(x_um.min()), float(x_um.max()), float(y_um.min()), float(y_um.max())]
    d_um = art["design_um"]

    fig, axes = plt.subplots(1, 2, figsize=(9.4, 4.1), constrained_layout=True)

    frame0 = eps_bg_base.copy()
    frame0[92:252, 92:252] = designs[0]
    im_eps = axes[0].imshow(
        frame0.T, origin="lower", extent=ext, cmap=cmap_mat, vmin=0.0, vmax=1.0, interpolation="nearest"
    )
    axes[0].add_patch(
        mpatches.Rectangle(
            (d_um[0], d_um[1]),
            d_um[2] - d_um[0],
            d_um[3] - d_um[1],
            fill=False,
            edgecolor="#d62728",
            lw=1.1,
            ls="--",
        )
    )
    axes[0].set_aspect("equal")
    axes[0].set_xlabel(r"$x$ [$\mu$m]")
    axes[0].set_ylabel(r"$y$ [$\mu$m]")
    axes[0].set_title("Round-2 `ceviche_waveguide_bend` Density")

    axes[1].plot(np.arange(n_total_iters), log_eval, "-", color="#cccccc", lw=1.4, label="Full 145-iter trace")
    (line_tr,) = axes[1].plot([0], [log_eval[0]], "-", color="#4f63d2", lw=2.0, label="Elapsed")
    (pt_cur,) = axes[1].plot([0], [log_eval[0]], "o", color="#e0549b", ms=5.5)
    axes[1].axhline(0.0, color="#666666", ls=":", lw=1.0)
    axes[1].axvline(104.5, color="#c62828", ls="--", lw=1.0, label="Enter STE (iter 105)")
    axes[1].set_xlim(0, n_total_iters - 1)
    axes[1].set_ylim(-2.5, 0.8)
    axes[1].set_xlabel("Optimization iteration")
    axes[1].set_ylabel("Wakis `eval_metric`")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(loc="lower right", fontsize=8.0)
    ttl = axes[1].set_title(f"Iter {kept_idx[0]} ({stages[0]}): eval = {log_eval[kept_idx[0]]:+.3f}", fontsize=9.2)

    def _update(frame_idx: int):
        it_k = int(kept_idx[frame_idx])
        f_map = eps_bg_base.copy()
        f_map[92:252, 92:252] = designs[frame_idx]
        im_eps.set_data(f_map.T)
        line_tr.set_data(np.arange(it_k + 1), log_eval[: it_k + 1])
        pt_cur.set_data([it_k], [log_eval[it_k]])
        ttl.set_text(f"Iter {it_k}/144 ({stages[frame_idx]}): eval = {log_eval[it_k]:+.3f}")
        return im_eps, line_tr, pt_cur, ttl

    anim = FuncAnimation(fig, _update, frames=n_frames, interval=140, blit=False)
    anim.save(fig_dir / "08_design_evolution.gif", writer=PillowWriter(fps=8), dpi=105)
    plt.close(fig)


def plot_all(fig_dir: Path | None = None) -> list[Path]:
    """Render all tutorial figures and setup schematics from ``data/round2_bend/`` and ``data/``."""
    target_dir = FIG_DIR if fig_dir is None else Path(fig_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    _plot_00_setup_schematics(target_dir)
    _plot_01_challenge_and_stack(target_dir)
    _plot_02_optimization_trace(target_dir)
    _plot_03_final_design_and_length_scale(target_dir)
    _plot_04_fields_six_wavelengths(target_dir)
    _plot_05_spectrum_vs_windows(target_dir)
    _plot_06_resim_and_fdfd_parity(target_dir)
    _plot_07_leaderboard_comparison(target_dir)
    _plot_08_design_evolution_gif(target_dir)
    return sorted(target_dir.glob("*"))

---

## Section 6 · Limits & Failure Modes, and Verified References

### Limits & Failure Modes

1. **Non-convex multi-seed spread (Claim C55) and global QCQP performance bounds**:
   Density-based photonic topology optimization is non-convex; different random
   initializations (`random_latent`) converge to distinct local minima on
   `ceviche_waveguide_bend` ($+0.5793$ for `ours_full_bend_seed1.npy` in
   `leaderboard_rescore.json` vs. $+0.6427$ in `round2_bend/summary.json`, a
   two-seed spread of $\Delta = 0.0634$). Differences smaller than $\sim 0.06$
   lie within run-to-run variance. To assess how close a local optimum comes to the
   physical limit of a $1.6\,\mu\text{m}\times 1.6\,\mu\text{m}$ silicon design region,
   Gertler, Kuang, Christie, Li & Miller (*Sci. Adv.* **11**, eadl3237, 2025) showed
   that differential-equation formulations of photonic waveguide design problems can
   be cast as sparse quadratically constrained quadratic programs (QCQPs) whose
   semidefinite relaxations yield rigorous upper bounds on achievable transmission.
2. **Stage-1 thresholding drop vs. Stage-2 morphological STE**:
   As recorded in `data/round2_bend/log.jsonl`, the continuous Stage-1 design
   reaches `eval_metric = +0.5585` at the end of `density_beta32` (step 19, iter 104),
   but its thresholded pattern has thin $3/5\text{ px}$ features; applying
   `imageruler` $8/8\text{ px}$ morphological repair immediately drops the score to
   `repaired_eval_metric = +0.1987` because expanding narrow features detunes the
   corner cavity phase. Running 40 steps of Stage-2 Adam STE (`ste_beta64`) on the
   repaired binary geometry recovers and surpasses the continuous score, reaching
   $+0.6412$ in Wakis ($+0.6427$ in `ceviche` FDFD) at step 34 while strictly
   maintaining $8/8\text{ px}$ at every iteration.
3. **40 nm lightweight optimization vs. 10 nm evaluation (Round-1 failure mode)**:
   In Round 1 (`data/round1/`), optimizing on the $40\text{ nm}$ lightweight grid
   without guaranteed morphological STE achieved $+0.3801$ (Seed 0) and $+0.5208$
   (Seed 1) at $40\text{ nm}$, but degraded to $-0.0355$ and $+0.3150$ when
   evaluated on the $10\text{ nm}$ challenge grid, with Seed 1 also violating the
   length scale ($2/1\text{ px}$ vs $3/3\text{ px}$ target). Optimizing directly on
   the $10\text{ nm}$ grid with `FeasibleDensity` + STE resolves both issues.
4. **Port-estimator reflection floor in FDFD cross-validation**:
   At $r = 1$ with `match_fdfd=True`, Wakis FIT and `ceviche` FDFD agree to
   $\max|\Delta|S|^2| = 3.77\times 10^{-4}$ across the 6 wavelengths of
   `ceviche_waveguide_bend` ($1.03\times 10^{-4}$ on the 40 nm lightweight bend).
   For back-reflection below $-31\text{ dB}$ ($|S_{11}|^2 \sim 10^{-5}$),
   `ceviche`'s two-column port overlap (`wakis_r1_ceviche_estimator_S2`) and
   Wakis's exact discrete `PolarizedModeMonitor` (`wakis_r1_modal_S2`) differ by
   $\sim 2\times 10^{-5}$ due to evanescent tail truncation at the finite port offset.

### Verified References

1. C. M. Lalau-Keraly, S. Bhargava, O. D. Miller and E. Yablonovitch, "Adjoint shape optimization applied to
   electromagnetic design", *Opt. Express* **21**, 21693 (2013),
   DOI: [`10.1364/OE.21.021693`](https://doi.org/10.1364/OE.21.021693).
2. M. Chen, R. E. Christiansen, J. A. Fan, G. Işiklar, J. Jiang, S. G. Johnson, W. Ma, O. D. Miller, A. Oskooi,
   M. F. Schubert, F. Wang, I. A. D. Williamson, W. Xue and Y. Zhou, "Validation and characterization of algorithms and
   software for photonics inverse design", *J. Opt. Soc. Am. B* **41**, A161 (2024),
   DOI: [`10.1364/JOSAB.506412`](https://doi.org/10.1364/JOSAB.506412).
3. L. Gertler, Z. Kuang, C. Christie, H. Li and O. D. Miller, "Many photonic design problems are sparse QCQPs",
   *Sci. Adv.* **11**, eadl3237 (2025),
   DOI: [`10.1126/sciadv.adl3237`](https://doi.org/10.1126/sciadv.adl3237).
4. M. F. Schubert, A. K. C. Cheung, I. A. D. Williamson, A. Spyra, and D. H. Alexander,
   "invrs-gym: a toolkit for nanophotonic inverse design research,"
   arXiv:[`2410.24132`](https://arxiv.org/abs/2410.24132) (2024).
5. T. W. Hughes, I. A. D. Williamson, M. Minkov, and S. Fan, "Forward-mode
   differentiation of Maxwell's equations," *ACS Photonics* **6**, 3010–3016 (2019),
   arXiv:[`1908.10507`](https://arxiv.org/abs/1908.10507),
   DOI: [`10.1021/acsphotonics.9b01238`](https://doi.org/10.1021/acsphotonics.9b01238).
6. F. Wang, B. S. Lazarov, and O. Sigmund, "On projection methods, convergence and
   robust formulations in topology optimization," *Structural and Multidisciplinary
   Optimization* **43**, 767–784 (2011),
   DOI: [`10.1007/s00158-010-0602-y`](https://doi.org/10.1007/s00158-010-0602-y).
7. B. Bourdin, "Filters in topology optimization," *International Journal for
   Numerical Methods in Engineering* **50**, 2143–2158 (2001),
   DOI: [`10.1002/nme.116`](https://doi.org/10.1002/nme.116).
8. S. Molesky, Z. Lin, A. Y. Piggott, W. Jin, J. Vučković, and A. W. Rodriguez,
   "Inverse design in nanophotonics," *Nature Photonics* **12**, 659–670 (2018),
   DOI: [`10.1038/s41566-018-0246-9`](https://doi.org/10.1038/s41566-018-0246-9).

In [ ]:
TASK_RUNNERS = {
    0: run_task_0_round2_trace_and_length_scale,
    1: run_task_1_wakis_resimulation,
    2: run_task_2_leaderboard_and_round1_comparison,
    3: run_task_3_toy_bend_teaching_optimization,
    4: run_task_4_optional_invrs_gym,
}


def run_tasks(quick: bool = False, task_index: int | None = None) -> dict[int, dict]:
    """Execute all tasks (or a single task by index) and return their summaries."""
    table = get_task_table(quick=quick)
    selected = table if task_index is None else [t for t in table if t["index"] == task_index]
    if not selected:
        raise ValueError(f"Unknown task index {task_index!r}; valid indices are 0..{len(table) - 1}.")
    summaries = {}
    for cfg in selected:
        idx = int(cfg["index"])
        t0 = time.perf_counter()
        res = TASK_RUNNERS[idx](cfg)
        dt_s = time.perf_counter() - t0
        summaries[idx] = {"name": cfg["name"], "elapsed_s": round(dt_s, 3), "summary": res}
        print(f"[task {idx}: {cfg['name']}] completed in {dt_s:.2f} s", flush=True)
    return summaries


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(
        description="Tutorial 9.2: Inverse design of a 90-degree waveguide bend (ceviche_waveguide_bend)."
    )
    mode = parser.add_mutually_exclusive_group()
    mode.add_argument("--quick", action="store_true", help="Run fast validation tasks (< 1 min).")
    mode.add_argument("--full", action="store_true", help="Run full-resolution 10 nm tasks.")
    parser.add_argument("--task", type=int, default=None, help="Run only task index I.")
    parser.add_argument("--out", type=Path, default=None, help="Override output data directory.")
    parser.add_argument("--plot", action="store_true", help="Regenerate figures from data/ only.")
    parser.add_argument("--list-tasks", action="store_true", help="Print task table as JSON and exit.")
    args = parser.parse_args(argv)

    global DATA_DIR
    if args.out is not None:
        DATA_DIR = Path(args.out)

    quick = not args.full
    if args.list_tasks:
        print(json.dumps(get_task_table(quick=quick), indent=2))
        return 0

    matplotlib.use("Agg")
    if args.plot:
        figs = plot_all()
        print(f"Generated {len(figs)} figures in {FIG_DIR}")
        return 0

    run_tasks(quick=quick, task_index=args.task)
    if args.task is None:
        figs = plot_all()
        print(f"Generated {len(figs)} figures in {FIG_DIR}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())